In [1]:

# imports
import os
import sys
import types
import json
import base64

# figure size/format
fig_width = 5.5
fig_height = 3.5
fig_format = 'pdf'
fig_dpi = 300
interactivity = ''
is_shiny = False
is_dashboard = False
plotly_connected = True

# matplotlib defaults / format
try:
  import matplotlib.pyplot as plt
  plt.rcParams['figure.figsize'] = (fig_width, fig_height)
  plt.rcParams['figure.dpi'] = fig_dpi
  plt.rcParams['savefig.dpi'] = "figure"

  # IPython 7.14 deprecated set_matplotlib_formats from IPython
  try:
    from matplotlib_inline.backend_inline import set_matplotlib_formats
  except ImportError:
    # Fall back to deprecated location for older IPython versions
    from IPython.display import set_matplotlib_formats
    
  set_matplotlib_formats(fig_format)
except Exception:
  pass

# plotly use connected mode
try:
  import plotly.io as pio
  if plotly_connected:
    pio.renderers.default = "notebook_connected"
  else:
    pio.renderers.default = "notebook"
  for template in pio.templates.keys():
    pio.templates[template].layout.margin = dict(t=30,r=0,b=0,l=0)
except Exception:
  pass

# disable itables paging for dashboards
if is_dashboard:
  try:
    from itables import options
    options.dom = 'fiBrtlp'
    options.maxBytes = 1024 * 1024
    options.language = dict(info = "Showing _TOTAL_ entries")
    options.classes = "display nowrap compact"
    options.paging = False
    options.searching = True
    options.ordering = True
    options.info = True
    options.lengthChange = False
    options.autoWidth = False
    options.responsive = True
    options.keys = True
    options.buttons = []
  except Exception:
    pass
  
  try:
    import altair as alt
    # By default, dashboards will have container sized
    # vega visualizations which allows them to flow reasonably
    theme_sentinel = '_quarto-dashboard-internal'
    def make_theme(name):
        nonTheme = alt.themes._plugins[name]    
        def patch_theme(*args, **kwargs):
            existingTheme = nonTheme()
            if 'height' not in existingTheme:
              existingTheme['height'] = 'container'
            if 'width' not in existingTheme:
              existingTheme['width'] = 'container'

            if 'config' not in existingTheme:
              existingTheme['config'] = dict()
            
            # Configure the default font sizes
            title_font_size = 15
            header_font_size = 13
            axis_font_size = 12
            legend_font_size = 12
            mark_font_size = 12
            tooltip = False

            config = existingTheme['config']

            # The Axis
            if 'axis' not in config:
              config['axis'] = dict()
            axis = config['axis']
            if 'labelFontSize' not in axis:
              axis['labelFontSize'] = axis_font_size
            if 'titleFontSize' not in axis:
              axis['titleFontSize'] = axis_font_size  

            # The legend
            if 'legend' not in config:
              config['legend'] = dict()
            legend = config['legend']
            if 'labelFontSize' not in legend:
              legend['labelFontSize'] = legend_font_size
            if 'titleFontSize' not in legend:
              legend['titleFontSize'] = legend_font_size  

            # The header
            if 'header' not in config:
              config['header'] = dict()
            header = config['header']
            if 'labelFontSize' not in header:
              header['labelFontSize'] = header_font_size
            if 'titleFontSize' not in header:
              header['titleFontSize'] = header_font_size    

            # Title
            if 'title' not in config:
              config['title'] = dict()
            title = config['title']
            if 'fontSize' not in title:
              title['fontSize'] = title_font_size

            # Marks
            if 'mark' not in config:
              config['mark'] = dict()
            mark = config['mark']
            if 'fontSize' not in mark:
              mark['fontSize'] = mark_font_size

            # Mark tooltips
            if tooltip and 'tooltip' not in mark:
              mark['tooltip'] = dict(content="encoding")

            return existingTheme
            
        return patch_theme

    # We can only do this once per session
    if theme_sentinel not in alt.themes.names():
      for name in alt.themes.names():
        alt.themes.register(name, make_theme(name))
      
      # register a sentinel theme so we only do this once
      alt.themes.register(theme_sentinel, make_theme('default'))
      alt.themes.enable('default')

  except Exception:
    pass

# enable pandas latex repr when targeting pdfs
try:
  import pandas as pd
  if fig_format == 'pdf':
    pd.set_option('display.latex.repr', True)
except Exception:
  pass

# interactivity
if interactivity:
  from IPython.core.interactiveshell import InteractiveShell
  InteractiveShell.ast_node_interactivity = interactivity

# NOTE: the kernel_deps code is repeated in the cleanup.py file
# (we can't easily share this code b/c of the way it is run).
# If you edit this code also edit the same code in cleanup.py!

# output kernel dependencies
kernel_deps = dict()
for module in list(sys.modules.values()):
  # Some modules play games with sys.modules (e.g. email/__init__.py
  # in the standard library), and occasionally this can cause strange
  # failures in getattr.  Just ignore anything that's not an ordinary
  # module.
  if not isinstance(module, types.ModuleType):
    continue
  path = getattr(module, "__file__", None)
  if not path:
    continue
  if path.endswith(".pyc") or path.endswith(".pyo"):
    path = path[:-1]
  if not os.path.exists(path):
    continue
  kernel_deps[path] = os.stat(path).st_mtime
print(json.dumps(kernel_deps))

# set run_path if requested
run_path = 'QzpcRGV2XGFyY2hpdmVcY2Fwc3RvbmUtcmstcmFqa3VtYXJcV2VlayAxMA=='
if run_path:
  # hex-decode the path
  run_path = base64.b64decode(run_path.encode("utf-8")).decode("utf-8")
  os.chdir(run_path)

# reset state
%reset

# shiny
# Checking for shiny by using False directly because we're after the %reset. We don't want
# to set a variable that stays in global scope.
if False:
  try:
    import htmltools as _htmltools
    import ast as _ast

    _htmltools.html_dependency_render_mode = "json"

    # This decorator will be added to all function definitions
    def _display_if_has_repr_html(x):
      try:
        # IPython 7.14 preferred import
        from IPython.display import display, HTML
      except:
        from IPython.core.display import display, HTML

      if hasattr(x, '_repr_html_'):
        display(HTML(x._repr_html_()))
      return x

    # ideally we would undo the call to ast_transformers.append
    # at the end of this block whenver an error occurs, we do 
    # this for now as it will only be a problem if the user 
    # switches from shiny to not-shiny mode (and even then likely
    # won't matter)
    import builtins
    builtins._display_if_has_repr_html = _display_if_has_repr_html

    class _FunctionDefReprHtml(_ast.NodeTransformer):
      def visit_FunctionDef(self, node):
        node.decorator_list.insert(
          0,
          _ast.Name(id="_display_if_has_repr_html", ctx=_ast.Load())
        )
        return node

      def visit_AsyncFunctionDef(self, node):
        node.decorator_list.insert(
          0,
          _ast.Name(id="_display_if_has_repr_html", ctx=_ast.Load())
        )
        return node

    ip = get_ipython()
    ip.ast_transformers.append(_FunctionDefReprHtml())

  except:
    pass

def ojs_define(**kwargs):
  import json
  try:
    # IPython 7.14 preferred import
    from IPython.display import display, HTML
  except:
    from IPython.core.display import display, HTML

  # do some minor magic for convenience when handling pandas
  # dataframes
  def convert(v):
    try:
      import pandas as pd
    except ModuleNotFoundError: # don't do the magic when pandas is not available
      return v
    if type(v) == pd.Series:
      v = pd.DataFrame(v)
    if type(v) == pd.DataFrame:
      j = json.loads(v.T.to_json(orient='split'))
      return dict((k,v) for (k,v) in zip(j["index"], j["data"]))
    else:
      return v

  v = dict(contents=list(dict(name=key, value=convert(value)) for (key, value) in kwargs.items()))
  display(HTML('<script type="ojs-define">' + json.dumps(v) + '</script>'), metadata=dict(ojs_define = True))
globals()["ojs_define"] = ojs_define
globals()["__spec__"] = None

{"C:\\Users\\karan\\AppData\\Roaming\\uv\\python\\cpython-3.12-windows-x86_64-none\\Lib\\importlib\\_bootstrap.py": 1772419517.7692616, "C:\\Users\\karan\\AppData\\Roaming\\uv\\python\\cpython-3.12-windows-x86_64-none\\Lib\\importlib\\_bootstrap_external.py": 1772419517.7712688, "C:\\Users\\karan\\AppData\\Roaming\\uv\\python\\cpython-3.12-windows-x86_64-none\\Lib\\zipimport.py": 1772419518.5049286, "C:\\Users\\karan\\AppData\\Roaming\\uv\\python\\cpython-3.12-windows-x86_64-none\\Lib\\codecs.py": 1772419517.5491636, "C:\\Users\\karan\\AppData\\Roaming\\uv\\python\\cpython-3.12-windows-x86_64-none\\Lib\\encodings\\aliases.py": 1772419517.6144166, "C:\\Users\\karan\\AppData\\Roaming\\uv\\python\\cpython-3.12-windows-x86_64-none\\Lib\\encodings\\__init__.py": 1772419517.6085927, "C:\\Users\\karan\\AppData\\Roaming\\uv\\python\\cpython-3.12-windows-x86_64-none\\Lib\\encodings\\utf_8.py": 1772419517.6738627, "C:\\Users\\karan\\AppData\\Roaming\\uv\\python\\cpython-3.12-windows-x86_64-none\

In [2]:
# 1. Environment Setup & Dependencies
# %pip install matplotlib seaborn scikit-learn shap pygam
import warnings
import re
import textwrap

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Markdown, display

# Modeling Libraries & Preprocessing
from sklearn.model_selection import train_test_split, RandomizedSearchCV, GridSearchCV
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text, _tree
from sklearn.neighbors import KNeighborsClassifier
from sklearn.utils.class_weight import compute_class_weight
from sklearn.feature_selection import SelectKBest, f_classif
from pygam import LogisticGAM, s, f  

# Metrics, Diagnostics & Explainability
from sklearn.metrics import recall_score, f1_score, confusion_matrix, classification_report, accuracy_score, ConfusionMatrixDisplay
from sklearn.inspection import permutation_importance
import shap

# --- Global Formatting Options ---
pd.set_option('display.max_columns', 10)      
pd.set_option('display.width', 70)            
pd.set_option('display.expand_frame_repr', True) 

# Load the Initial Dataset
ibm = pd.read_csv("ibm.csv")

In [3]:
# 2. MARS Feature Extraction
mars_vars = [
    # Target Variable
    'Attrition',
    
    # Motivation 
    'JobSatisfaction', 
    'TrainingTimesLastYear', 
    'StockOptionLevel',
    
    # Ability 
    'PerformanceRating', 
    'Education', 
    'EducationField', 
    'YearsAtCompany',
    
    # Role Perception
    'Department', 
    'JobLevel', 
    'YearsWithCurrManager', 
    'YearsInCurrentRole',
    
    # Situational Factors 
    'DistanceFromHome', 
    'OverTime', 
    'BusinessTravel', 
    'EnvironmentSatisfaction'
]

# Create the specific dataframe for analysis
ibm_trunc = ibm[mars_vars].copy()


In [4]:
# 3. Categorical Encoding, Scaling & Train-Test Split (IBM Dataset)

# Binary encode Attrition and OverTime
binary_dict = {'Yes': 1, 'No': 0}
ibm_trunc.Attrition = ibm_trunc.Attrition.replace(binary_dict)
ibm_trunc.OverTime = ibm_trunc.OverTime.replace(binary_dict)

# Manually encode ordinal/low-cardinality categorical variable (BusinessTravel)
travel_dict = dict(zip(ibm_trunc.BusinessTravel.unique(), [1, 2, 0]))
ibm_trunc.BusinessTravel = ibm_trunc.BusinessTravel.replace(travel_dict)

# Separate Dependent (y) and Independent (X) variables
ibm_y = ibm_trunc.Attrition.values
ibm_X_vals = ibm_trunc.drop(['Attrition'], axis=1)

# One-hot encode the remaining nominal categorical columns (EducationField, Department)
ibm_X_vals = pd.get_dummies(ibm_X_vals)

# --- Append MARS Tags to Feature Names ---
# We map the base features to their respective MARS categories based on your Section 2 definitions
mars_tag_map = {
    'JobSatisfaction': '(M)', 'TrainingTimesLastYear': '(M)', 'StockOptionLevel': '(M)',
    'PerformanceRating': '(A)', 'Education': '(A)', 'EducationField': '(A)', 'YearsAtCompany': '(A)',
    'Department': '(R)', 'JobLevel': '(R)', 'YearsWithCurrManager': '(R)', 'YearsInCurrentRole': '(R)',
    'DistanceFromHome': '(S)', 'OverTime': '(S)', 'BusinessTravel': '(S)', 'EnvironmentSatisfaction': '(S)'
}

ibm_X_labels = []
for col in ibm_X_vals.columns:
    tag = ''
    for base_feature, mars_tag in mars_tag_map.items():
        # Using startswith() ensures one-hot encoded columns (like 'Department_Sales') catch the base tag
        if col.startswith(base_feature):
            tag = mars_tag
            break
    ibm_X_labels.append(f"{col} {tag}")

# Assign the updated labels back to the dataframe and extract values
ibm_X_vals.columns = ibm_X_labels
ibm_X = ibm_X_vals.values

# Encoding target as integers to avoid model dtype warnings later
le = LabelEncoder()
ibm_y = le.fit_transform(ibm_y)

# MinMax scaling needed since SVM and kNN rely on distance metrics
scaling_obj = MinMaxScaler()
ibm_X_sc = scaling_obj.fit_transform(ibm_X)

# Create unscaled train/test splits (Used for Decision Trees)
ibm_X_train, ibm_X_test, ibm_y_train, ibm_y_test = train_test_split(
    ibm_X, ibm_y, test_size=0.15, random_state=16, stratify=ibm_y
)

# Create scaled train/test splits (Used for Linear Models, SVM, and kNN)
# The unscaled y variables are safely reused here because class labels do not change during feature scaling
ibm_X_train_sc, ibm_X_test_sc = train_test_split(
    ibm_X_sc, ibm_y, test_size=0.15, random_state=16, stratify=ibm_y
)[:2]


In [5]:
def get_top_kill_zones(tree_model, feature_names, target_names, n_zones=3):
    tree_ = tree_model.tree_
    feature_name = [
        feature_names[i] if i != _tree.TREE_UNDEFINED else "undefined!"
        for i in tree_.feature
    ]

    paths = []
    
    # Extract the raw, unweighted headcount of the entire training dataset
    raw_total_headcount = tree_.n_node_samples[0]

    def recurse(node, path):
        if tree_.feature[node] != _tree.TREE_UNDEFINED:
            name = feature_name[node]
            threshold = tree_.threshold[node]
            # Left child (True condition: <= threshold)
            recurse(tree_.children_left[node], path + [f"{name} <= {threshold:.2f}"])
            # Right child (False condition: > threshold)
            recurse(tree_.children_right[node], path + [f"{name} > {threshold:.2f}"])
        else:
            # Terminal Leaf: Calculate concentration using weighted scores
            val = tree_.value[node][0]
            attrition_count = val[1]  # Class 1 is Attrition
            total_count = sum(val)
            
            # Extract the raw, unweighted headcount for this specific leaf
            raw_leaf_headcount = tree_.n_node_samples[node]
            
            # Avoid division by zero
            if total_count > 0:
                attrition_rate = attrition_count / total_count
                
                # Calculate what percentage of the raw workforce falls into this leaf
                workforce_pct = (raw_leaf_headcount / raw_total_headcount) * 100
                
                paths.append((attrition_rate, attrition_count, total_count, raw_leaf_headcount, workforce_pct, path))

    recurse(0, [])
    
    # Sort paths by highest attrition rate first
    top_paths = sorted(paths, key=lambda x: x[0], reverse=True)[:n_zones]

    print(f"=== Top {n_zones} Attrition Kill Zones Found ===")
    for i, (rate, count, total, raw_hc, wpct, path) in enumerate(top_paths, 1):
        print(f"\nKill Zone {i}: {rate:.1%} Attrition Concentration")
        print(f"Impact: {count:.1f}/{total:.1f} weighted score (Applies to {raw_hc} actual employees | {wpct:.2f}% of workforce)")
        
        # Wrapping the text at 85 chars so it doesn't run off the right side of the PDF
        logic_string = "Logic Path: " + " AND ".join(path)
        print(textwrap.fill(logic_string, width=85))

In [6]:
# An empty dictionary is created to store the results of model evaluations
results_dict = {
    'model name': [],
    'tuned hyperparameters': [],
    'specificity': [],
    'sensitivity': [],
    'f1 score': []
}

test_labels = ['Retained', 'Terminated']

def classification_summary(y_test, y_pred, model_name, tuned_params="Baseline (Untuned)", show_visuals=False):
    """
    Takes in predicted and true labels, appends metrics to a global dictionary,
    and optionally produces a Confusion Matrix diagram and classification summary.
    
    Args:
        y_test (ndarray): True labels for the test set
        y_pred (ndarray): Predicted labels for the test set
        model_name (str): The name of the model for storage and display
        tuned_params (dict or str): Optimal parameters from grid search or manual input
        show_visuals (bool): Toggle to print the classification report and plot the confusion matrix.
    """
    # Calculate metrics
    cm = confusion_matrix(y_test, y_pred)
    cr = classification_report(y_test, y_pred, output_dict=True)
    
    # Clean parameter formatting for the final DataFrame
    if isinstance(tuned_params, dict):
        # We strip the bulky 'class_weight' dictionary out so it doesn't wreck the pandas table formatting
        clean_params = {k: v for k, v in tuned_params.items() if k != 'class_weight'}
        param_str = str(clean_params)
    else:
        # Handles manual string inputs like "k=5" for kNN
        param_str = str(tuned_params)
        
    # Append the model performance results to results_dict
    results_dict['model name'].append(model_name)  
    results_dict['tuned hyperparameters'].append(param_str)
    results_dict['specificity'].append(cr['0']['recall'])
    results_dict['sensitivity'].append(cr['1']['recall'])
    results_dict['f1 score'].append(cr['weighted avg']['f1-score'])

    # Toggle visual output
    if show_visuals:
        cr_display = classification_report(y_test, y_pred)
        conf_vis = ConfusionMatrixDisplay(cm, display_labels=test_labels)
        
       # print(f'\n--- Evaluation for {model_name} ---')
      #  print(f'Hyperparameters: {param_str}')
      #  print(f'The confusion matrix for the {model_name} model is:')
        
        fig, ax = plt.subplots(figsize=(4.8, 3.2))
        conf_vis.plot(cmap='magma_r', ax=ax)
        plt.show()
        
        print(f'The classification report for the {model_name} model is:')
        print(cr_display)

In [7]:
# --- CLEAN PRE-PROCESSING CONSOLE LOG ---

# Compute raw weights
ibm_raw_weights = compute_class_weight(y=ibm_y, class_weight='balanced', classes=np.unique(ibm_y))

# Converting numpy floats to native python types so it prints cleanly
ibm_weights_dict = {int(cls): round(float(weight), 2) for cls, weight in zip(np.unique(ibm_y), ibm_raw_weights)}

# Single-line consolidated print statement

print(f"IBM Dataset Pipeline | Shape Filtered: {ibm.shape} -> {ibm_trunc.shape}\nComputed Imbalance Weights: {ibm_weights_dict}")

print("\nFinal MARS-Tagged Features:") 
for label in ibm_X_labels:
    print(f" - {label}")


IBM Dataset Pipeline | Shape Filtered: (1470, 35) -> (1470, 16)
Computed Imbalance Weights: {0: 0.6, 1: 3.1}

Final MARS-Tagged Features:
 - JobSatisfaction (M)
 - TrainingTimesLastYear (M)
 - StockOptionLevel (M)
 - PerformanceRating (A)
 - Education (A)
 - YearsAtCompany (A)
 - JobLevel (R)
 - YearsWithCurrManager (R)
 - YearsInCurrentRole (R)
 - DistanceFromHome (S)
 - EnvironmentSatisfaction (S)
 - EducationField_Human Resources (A)
 - EducationField_Life Sciences (A)
 - EducationField_Marketing (A)
 - EducationField_Medical (A)
 - EducationField_Other (A)
 - EducationField_Technical Degree (A)
 - Department_Human Resources (R)
 - Department_Research & Development (R)
 - Department_Sales (R)
 - OverTime_0 (S)
 - OverTime_1 (S)
 - BusinessTravel_0 (S)
 - BusinessTravel_1 (S)
 - BusinessTravel_2 (S)


In [8]:
#| label: fig-ibm-cmatrix
#| fig-cap: Confusion Matrix for Std Logistic Regression Model (IBM)

# --- Model A: Standard Logistic Regression ---
ibm_lr_classifier = LogisticRegression(random_state=12)

ibm_lr_classifier.fit(ibm_X_train_sc, ibm_y_train)

ibm_y_pred_lr = ibm_lr_classifier.predict(ibm_X_test_sc)

classification_summary(ibm_y_test, ibm_y_pred_lr, 'A: Standard Logistic Regression', show_visuals=True)

<Figure size 1440x960 with 2 Axes>

The classification report for the A: Standard Logistic Regression model is:
              precision    recall  f1-score   support

           0       0.87      0.99      0.92       185
           1       0.80      0.22      0.35        36

    accuracy                           0.86       221
   macro avg       0.83      0.61      0.64       221
weighted avg       0.86      0.86      0.83       221



In [9]:
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    
    # --- Model B: Regularized Logistic Regression ---
    ibm_logreg_params = {
        "C": np.linspace(0.001, 1000, 10),
        "solver": ['newton-cg', 'saga'],
        "penalty": ['l2'],
        "class_weight": [ibm_weights_dict, None]
    }
    ibm_logreg = LogisticRegression(max_iter=10000)

    ibm_logreg_grid = RandomizedSearchCV(ibm_logreg, ibm_logreg_params, random_state=6, n_iter=20, scoring='f1')
    ibm_logreg_grid.fit(ibm_X_train_sc, ibm_y_train)

    ibm_l2logreg_model = LogisticRegression(**ibm_logreg_grid.best_params_)
    ibm_l2logreg_model.fit(ibm_X_train_sc, ibm_y_train)
    ibm_y_pred_l2_logreg = ibm_l2logreg_model.predict(ibm_X_test_sc)

    classification_summary(
        ibm_y_test, 
        ibm_y_pred_l2_logreg, 
        'B: Regularized Logistic Regression', 
        tuned_params=ibm_logreg_grid.best_params_
    )

    # --- Model C: Linear SVM ---
    ibm_svm_params = {
        "C": np.logspace(-3, 3, 20),
        "class_weight": [ibm_weights_dict]
    }
    ibm_svm = LinearSVC(penalty='l2', loss='squared_hinge', dual=False, max_iter=20000)

    ibm_svm_grid = RandomizedSearchCV(ibm_svm, ibm_svm_params, random_state=21, scoring='f1')
    ibm_svm_grid.fit(ibm_X_train_sc, ibm_y_train)

    ibm_best_svm = ibm_svm_grid.best_estimator_
    ibm_y_pred_svm = ibm_best_svm.predict(ibm_X_test_sc)

    classification_summary(
        ibm_y_test, 
        ibm_y_pred_svm, 
        'C: Linear SVM (Tuned)', 
        tuned_params=ibm_svm_grid.best_params_
    )

    # --- Model D: Decision Tree ---
    ibm_tree_params = {
        "max_depth": np.arange(3, 10, 2),
        "min_samples_split": np.arange(2, 9, 1),
        "min_samples_leaf": np.arange(20, 50, 10),
        "criterion": ['gini', 'log_loss'],
        "class_weight": [ibm_weights_dict, None]
    }
    ibm_tree = DecisionTreeClassifier()

    ibm_tree_grid = RandomizedSearchCV(ibm_tree, ibm_tree_params, random_state=6, n_iter=50, scoring='f1')
    # Note: Decision Trees use the unscaled features (ibm_X_train)
    ibm_tree_grid.fit(ibm_X_train, ibm_y_train)

    ibm_tree_model = DecisionTreeClassifier(**ibm_tree_grid.best_params_)
    ibm_tree_model.fit(ibm_X_train, ibm_y_train)

    ibm_y_pred_dt = ibm_tree_model.predict(ibm_X_test)

    # If you decide to use Model D as your visual "textbook example" for the naive reader, 
    # you can easily switch show_visuals=True here later.
    classification_summary(
        ibm_y_test, 
        ibm_y_pred_dt, 
        'D: Decision Tree', 
        tuned_params=ibm_tree_grid.best_params_
    )

In [10]:
# --- Model F: Generalized Additive Model (GAM) [OPTIMIZED] ---
ibm_gam_sample_weights = np.array([ibm_weights_dict[cls] for cls in ibm_y_train])

# OPTIMIZATION: Shrink the search space from 11 lambdas to 5 to drastically speed up render time
ibm_gam_lams = np.logspace(-2, 2, 5)

# Dropping n_splines to 10 to speed up execution and prevent overfitting the noise
ibm_gam_model = LogisticGAM(n_splines=10)

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    ibm_gam_model.gridsearch(
        ibm_X_train_sc, 
        ibm_y_train, 
        lam=ibm_gam_lams, 
        weights=ibm_gam_sample_weights, 
        progress=False
    )

ibm_y_pred_gam = ibm_gam_model.predict(ibm_X_test_sc)

classification_summary(
    ibm_y_test, 
    ibm_y_pred_gam, 
    'F: Generalized Additive Model (GAM)', 
    tuned_params={'lam': round(float(ibm_gam_model.lam[0][0]), 3)} 
)

In [11]:
#| label: fig-ibm-knn
#| fig-cap: 'KNN Elbow Diagram: Attrition Recall vs. Neighbors (IBM Dataset)'

# Initialize variables for the Elbow Diagram...

# We use f_classif which handles the strictly positive values from MinMaxScaler
ibm_selector = SelectKBest(score_func=f_classif, k=10)
ibm_X_train_selected = ibm_selector.fit_transform(ibm_X_train_sc, ibm_y_train)
ibm_X_test_selected = ibm_selector.transform(ibm_X_test_sc)

ibm_selected_mask = ibm_selector.get_support()
ibm_selected_features = np.array(ibm_X_labels)[ibm_selected_mask]

print(f"kNN will now use only these {len(ibm_selected_features)} features:")
print(ibm_selected_features)

# Initialize variables for the Elbow Diagram
ibm_neighbors = range(1, 20)
ibm_knn_train_recall = {}
ibm_knn_test_recall = {}

for neighbor in ibm_neighbors:
    # Initialize and Fit 
    # Added weights='distance' to prevent the majority class from swamping the vote
    ibm_knn = KNeighborsClassifier(n_neighbors=neighbor, weights='distance')
    ibm_knn.fit(ibm_X_train_selected, ibm_y_train)
    
    # Predict
    ibm_y_pred_train_knn = ibm_knn.predict(ibm_X_train_selected)
    ibm_y_pred_test_knn = ibm_knn.predict(ibm_X_test_selected)
    
    # Calculate RECALL for the Positive Class (Attrition)
    ibm_knn_train_recall[neighbor] = recall_score(ibm_y_train, ibm_y_pred_train_knn)
    ibm_knn_test_recall[neighbor] = recall_score(ibm_y_test, ibm_y_pred_test_knn)

# Plotting the Elbow Diagram anchored on Recall
plt.figure(figsize=(10, 6))
plt.title("KNN Elbow Diagram: Attrition Recall vs Neighbors (IBM Dataset)")
plt.plot(ibm_neighbors, list(ibm_knn_train_recall.values()), label="Training Recall", marker='o')
plt.plot(ibm_neighbors, list(ibm_knn_test_recall.values()), label="Testing Recall", marker='o')
plt.legend()
plt.xlabel("Number of Neighbors (k)")
plt.ylabel("Recall (Sensitivity for Class 1: Attrition)")
plt.grid(True, linestyle='--', alpha=0.7)
plt.xticks(ibm_neighbors) 
plt.show()

# Extract the exact best k dynamically based purely on max Test Recall
ibm_best_k = max(ibm_knn_test_recall, key=ibm_knn_test_recall.get)
print(f"Peak Performance: k={ibm_best_k} with Test Recall of {ibm_knn_test_recall[ibm_best_k]:.4f}")

# Train the finalized KNN model with the optimal k
ibm_knn_optimized = KNeighborsClassifier(n_neighbors=ibm_best_k, weights='distance')
ibm_knn_optimized.fit(ibm_X_train_selected, ibm_y_train)
ibm_y_pred_knn = ibm_knn_optimized.predict(ibm_X_test_selected)

# Pass the final results to the silent summary tracker
classification_summary(
    ibm_y_test, 
    ibm_y_pred_knn, 
    f'E: k-Nearest Neighbors', 
    tuned_params=f"n_neighbors={ibm_best_k}"
)

kNN will now use only these 10 features:
['JobSatisfaction (M)' 'StockOptionLevel (M)' 'YearsAtCompany (A)'
 'JobLevel (R)' 'YearsWithCurrManager (R)' 'YearsInCurrentRole (R)'
 'EnvironmentSatisfaction (S)' 'OverTime_0 (S)' 'OverTime_1 (S)'
 'BusinessTravel_2 (S)']


<Figure size 3000x1800 with 1 Axes>

Peak Performance: k=7 with Test Recall of 0.4167


In [12]:
# --- REFACTORED GLOBAL STORAGE INITIALIZATION ---


# Initialize all THREE global trackers
ibm_global_feature_importances = pd.DataFrame(columns=['Model', 'Feature', 'Importance', 'Raw_Value', 'Color'])
bab_global_feature_importances = pd.DataFrame(columns=['Model', 'Feature', 'Importance', 'Raw_Value', 'Color'])
ds_global_feature_importances = pd.DataFrame(columns=['Model', 'Feature', 'Importance', 'Raw_Value', 'Color'])

# --- REFACTORED INGESTION FUNCTION ---
def store_feature_importances(labels, importances, model_name, is_directional=True, dataset='IBM'):
    """
    Evaluates feature magnitude/direction and appends it to the global tracker.
    Uses default kwargs to maintain backward compatibility with previous IBM code.
    Accepted datasets: 'IBM', 'Babushkin', 'DS_Job'
    """
    global ibm_global_feature_importances 
    global bab_global_feature_importances
    global ds_global_feature_importances

    # 1. Determine Colors
    if is_directional:
        colors = ['red' if val > 0 else 'blue' for val in importances]
    else:
        colors = ['grey'] * len(importances) 

    # 2. Create local dataframe and force strict float types
    local_df = pd.DataFrame({
        'Model': model_name,
        'Feature': labels,
        'Importance': np.abs(importances).astype(float), 
        'Raw_Value': np.array(importances).astype(float),          
        'Color': colors         
    })
    
    # 3. Append to the correct Global Dataframe
    if dataset == 'IBM':
        ibm_global_feature_importances = pd.concat([ibm_global_feature_importances, local_df], ignore_index=True)
    elif dataset == 'Babushkin':
        bab_global_feature_importances = pd.concat([bab_global_feature_importances, local_df], ignore_index=True)
    elif dataset == 'DS_Job':
        ds_global_feature_importances = pd.concat([ds_global_feature_importances, local_df], ignore_index=True)
    else:
        raise ValueError("Invalid dataset name provided.")

In [13]:
# --- POPULATE THE GLOBAL VIMS DATAFRAME ---
# Model A: Standard Logistic Regression (Directional)
store_feature_importances(ibm_X_labels, ibm_lr_classifier.coef_[0], 'A: Standard Logistic Regression', is_directional=True)

# Model B: Regularized Ridge (Directional)
store_feature_importances(ibm_X_labels, ibm_l2logreg_model.coef_[0], 'B: Regularized Logistic Regression', is_directional=True)

# Model C: Linear SVM (Directional)
store_feature_importances(ibm_X_labels, ibm_best_svm.coef_[0], 'C: Linear SVM (Tuned)', is_directional=True)

# Model D: Decision Tree (Magnitude Only)
store_feature_importances(ibm_X_labels, ibm_tree_model.feature_importances_, 'D: Decision Tree', is_directional=False)

# Model E: kNN (Magnitude Only via ANOVA F-Scores)
store_feature_importances(ibm_X_labels, ibm_selector.scores_, f'E: kNN (k={ibm_best_k})', is_directional=False)

# Model F: Generalized Additive Model (Magnitude Only via Permutation Importance)
# We shuffle the test set features to see which ones cause the biggest drop in accuracy
ibm_gam_perm_imp = permutation_importance(ibm_gam_model, ibm_X_test_sc, ibm_y_test, n_repeats=5, random_state=42)
store_feature_importances(ibm_X_labels, ibm_gam_perm_imp.importances_mean, 'F: Generalized Additive Model (GAM)', is_directional=False)

In [14]:
#| label: fig-ibm-vims
#| fig-cap: Global Feature Importance by Algorithm (IBM Dataset)

# --- REFACTORED MASTER VISUAL FUNCTION ---

def plot_vims_facet_grid(dataset='IBM'):
    global ibm_global_feature_importances
    global bab_global_feature_importances
    global ds_global_feature_importances
    
    # Select the correct dataframe to plot based on the argument
    if dataset == 'IBM':
        target_df = ibm_global_feature_importances
    elif dataset == 'Babushkin':
        target_df = bab_global_feature_importances
    elif dataset == 'DS_Job':
        target_df = ds_global_feature_importances
    else:
        raise ValueError("Invalid dataset name provided.")
        
    # Force the Importance column to be numeric to prevent nlargest() crashes
    target_df['Importance'] = pd.to_numeric(target_df['Importance'])
    
    # Create the 3x2 grid
    fig, axes = plt.subplots(nrows=3, ncols=2, figsize=(14, 18), sharex=False)
    axes = axes.flatten()
    
    # Grab the unique models currently stored in the tracker
    models = target_df['Model'].unique()
    
    for i, model in enumerate(models):
        ax = axes[i]
        model_data = target_df[target_df['Model'] == model]
        top_10 = model_data.nlargest(10, 'Importance').sort_values('Importance', ascending=True)
        
        ax.barh(top_10['Feature'], top_10['Importance'], color=top_10['Color'], edgecolor='black', linewidth=0.5)
        
        ax.set_title(model, fontsize=14, fontweight='bold')
        ax.tick_params(axis='y', labelsize=10)
        ax.xaxis.grid(True, linestyle='--', alpha=0.6)
        ax.set_axisbelow(True)

    # Hide the 6th empty subplot if GAM isn't ingested yet
    if len(models) < 6:
        axes[5].set_visible(False)
        
    # Dynamic Super Title updates automatically based on the dataset argument
    fig.suptitle(f'Global Feature Importance by Algorithm ({dataset} Dataset)\nRed = Attrition Driver | Blue = Retention Factor | Grey = Magnitude Only', fontsize=16, y=1.02)
    
    plt.tight_layout()
    plt.show()

plot_vims_facet_grid()

<Figure size 4200x5400 with 6 Axes>

In [15]:
#| label: fig-ibm-pdp
#| fig-cap: Non-Linear Feature Effects for Top Continuous Drivers (IBM GAM)

# --- MODEL F: PARTIAL DEPENDENCE PLOTS (STRIPPED SYNTAX) ---

# 1. Extract Top 3 Features
ibm_gam_vims = ibm_global_feature_importances[ibm_global_feature_importances['Model'] == 'F: Generalized Additive Model (GAM)']
ibm_top_3_gam_features = ibm_gam_vims.nlargest(3, 'Importance')['Feature'].tolist()
ibm_top_3_indices = [ibm_X_labels.index(feat) for feat in ibm_top_3_gam_features]

# 2. Create the 1x3 Grid
# Changed to 3 rows, 1 column. Adjusted figsize for vertical stacking.
fig, axes = plt.subplots(nrows=3, ncols=1, figsize=(10, 15))

# (Keep your existing for-loop exactly the same, but remove the `if i == 0:` y-axis 
# logic since every plot is now on the left edge and needs a y-axis label)

for i, (feat, idx) in enumerate(zip(ibm_top_3_gam_features, ibm_top_3_indices)):
    ax = axes[i]
    
    # Generate the grid for the specific term
    XX = ibm_gam_model.generate_X_grid(term=idx)
    
    # Calculate dependence - keep 'term' here as partial_dependence requires it
    pdp = ibm_gam_model.partial_dependence(term=idx, X=XX)
    
    # The model infers the term from the structure of XX. 
    confi = ibm_gam_model.confidence_intervals(XX, width=0.95)
    
    # Plotting
    ax.plot(XX[:, idx], pdp, color='purple', linewidth=2)
    ax.fill_between(XX[:, idx], confi[:, 0], confi[:, 1], color='purple', alpha=0.2)
    
    # Baseline
    ax.axhline(0, color='black', linestyle='--', linewidth=1.1)
    
    # Formatting
    ax.set_title(f"PDP: {feat}", fontsize=12, fontweight='bold')
    ax.set_xlabel("Scaled Value (0 to 1)")
    if i == 0:
        ax.set_ylabel("Partial Dependence (Log-Odds)")
    
    ax.grid(True, linestyle=':', alpha=0.6)

fig.suptitle('Non-Linear Feature Effects: Top 3 IBM Drivers (GAM)', fontsize=15, y=1.05)
plt.tight_layout()
plt.show()

<Figure size 3000x4500 with 3 Axes>

In [16]:
get_top_kill_zones(ibm_tree_model, ibm_X_labels, ['Stay', 'Exit'])

=== Top 3 Attrition Kill Zones Found ===

Kill Zone 1: 92.8% Attrition Concentration
Impact: 0.9/1.0 weighted score (Applies to 35 actual employees | 2.80% of workforce)
Logic Path: OverTime_0 (S) <= 0.50 AND JobLevel (R) <= 1.50 AND YearsInCurrentRole
(R) <= 0.50

Kill Zone 2: 90.6% Attrition Concentration
Impact: 0.9/1.0 weighted score (Applies to 20 actual employees | 1.60% of workforce)
Logic Path: OverTime_0 (S) <= 0.50 AND JobLevel (R) <= 1.50 AND YearsInCurrentRole
(R) > 0.50 AND DistanceFromHome (S) > 12.50

Kill Zone 3: 89.4% Attrition Concentration
Impact: 0.9/1.0 weighted score (Applies to 21 actual employees | 1.68% of workforce)
Logic Path: OverTime_0 (S) <= 0.50 AND JobLevel (R) > 1.50 AND Department_Sales (R) >
0.50 AND StockOptionLevel (M) <= 0.50 AND DistanceFromHome (S) > 5.50


In [17]:
# --- PREP DATASET FOR SHAP ---
# Re-attaching labels for readability
ibm_X_test_df = pd.DataFrame(ibm_X_test_sc, columns=ibm_X_labels)
ibm_X_train_df = pd.DataFrame(ibm_X_train_sc, columns=ibm_X_labels)

# --- 1. MODEL A: Standard Logistic Regression ---
ibm_probas_a = ibm_lr_classifier.predict_proba(ibm_X_test_sc)[:, 1]
idx_a_min, idx_a_max = np.argmin(ibm_probas_a), np.argmax(ibm_probas_a)
ibm_explainer_a = shap.LinearExplainer(ibm_lr_classifier, ibm_X_train_df)
ibm_shap_a = ibm_explainer_a(ibm_X_test_df)

# --- 2. MODEL B: Regularized Logistic Regression ---
ibm_probas_b = ibm_l2logreg_model.predict_proba(ibm_X_test_sc)[:, 1]
idx_b_min, idx_b_max = np.argmin(ibm_probas_b), np.argmax(ibm_probas_b)
ibm_explainer_b = shap.LinearExplainer(ibm_l2logreg_model, ibm_X_train_df)
ibm_shap_b = ibm_explainer_b(ibm_X_test_df)

# --- 3. MODEL C: Linear SVM ---
ibm_scores_c = ibm_best_svm.decision_function(ibm_X_test_sc)
idx_c_min, idx_c_max = np.argmin(ibm_scores_c), np.argmax(ibm_scores_c)
ibm_explainer_c = shap.LinearExplainer(ibm_best_svm, ibm_X_train_df)
ibm_shap_c = ibm_explainer_c(ibm_X_test_df)

# --- 4. MODEL F: Generalized Additive Model (GAM) ---
ibm_probas_f = ibm_gam_model.predict_proba(ibm_X_test_sc)
idx_f_min, idx_f_max = np.argmin(ibm_probas_f), np.argmax(ibm_probas_f)

# We use a smaller background sample for GAM SHAP to speed up the kernel calculation
ibm_gam_bg = shap.sample(ibm_X_train_df, 50) 
ibm_explainer_f = shap.Explainer(ibm_gam_model.predict, ibm_gam_bg)

# Slicing just the min/max rows here, otherwise KernelExplainer takes forever to run
ibm_shap_f_safe = ibm_explainer_f(ibm_X_test_df.iloc[[idx_f_min]], silent=True)
ibm_shap_f_risk = ibm_explainer_f(ibm_X_test_df.iloc[[idx_f_max]], silent=True)

In [18]:
#| layout-ncol: 2
#| label: fig-ibm-shap-safe
#| fig-cap: 'SHAP Local Interpretability: Profiles with Lowest Attrition Probability (IBM)'
#| fig-subcap:
#|   - 'A: Std Logistic Regression - Safest'
#|   - 'B: Reg. Logistic Regression - Safest'
#|   - 'C: Linear SVM (Tuned) - Safest'
#|   - 'F: GAM - Safest'

def render_shap_waterfall(shap_vals, plot_title):
    # Base font size for y-axis labels
    plt.rcParams.update({'font.size': 14})
    
    shap.plots.waterfall(shap_vals, max_display=8, show=False)
    
    fig = plt.gcf()
    ax = plt.gca()
    
    # STRETCHED canvas (8x9) to perfectly fill the vertical page height
    fig.set_size_inches(8, 8) 
    
    # Targeted text shrink for the inner bar magnitudes
    for text in ax.texts:
        t = text.get_text()
        if t.startswith('+') or t.startswith('-'):
            text.set_fontsize(11)
            text.set_weight('bold')
    
    plt.title(plot_title, pad=20, fontweight='bold', fontsize=18)
    plt.tight_layout()
    plt.show()
    
    plt.rcParams.update({'font.size': 10})

# --- THE "SAFE" ANCHORS (Lowest Probabilities) ---
render_shap_waterfall(ibm_shap_a[idx_a_min], "")
render_shap_waterfall(ibm_shap_b[idx_b_min], "")
render_shap_waterfall(ibm_shap_c[idx_c_min], "")
render_shap_waterfall(ibm_shap_f_safe[0], "")

<Figure size 2400x2400 with 3 Axes>

<Figure size 2400x2400 with 3 Axes>

<Figure size 2400x2400 with 3 Axes>

<Figure size 2400x2400 with 3 Axes>

In [19]:
#| layout-ncol: 2
#| label: fig-ibm-shap-risk
#| fig-cap: 'SHAP Local Interpretability: Profiles with Highest Attrition Probability (IBM)'
#| fig-subcap:
#|   - 'A: Std Logistic Regression - Riskiest'
#|   - 'B: Reg. Logistic Regression - Riskiest'
#|   - 'C: Linear SVM (Tuned) - Riskiest'
#|   - 'F: GAM - Riskiest'

# --- THE "FLIGHT RISKS" (Highest Probabilities) ---

# (We reuse the render_shap_waterfall function we defined in the previous cell)

render_shap_waterfall(ibm_shap_a[idx_a_max], "")
render_shap_waterfall(ibm_shap_b[idx_b_max], "")
render_shap_waterfall(ibm_shap_c[idx_c_max], "")
render_shap_waterfall(ibm_shap_f_risk[0], "")

<Figure size 2400x2400 with 3 Axes>

<Figure size 2400x2400 with 3 Axes>

<Figure size 2400x2400 with 3 Axes>

<Figure size 2400x2400 with 3 Axes>

In [20]:
#| label: tbl-ibm-perf
#| tbl-cap: Consolidated Algorithm Performance and Hyperparameters (IBM Dataset)

# 4. Universal Performance Matrix (IBM Dataset)

# Convert the populated results dictionary into a master dataframe
ibm_performance_df = pd.DataFrame(results_dict)

# Using regex to strip out numpy wrappers from the hyperparameter strings so the table looks clean
ibm_performance_df['tuned hyperparameters'] = ibm_performance_df['tuned hyperparameters'].apply(
    lambda x: re.sub(r"np\.\w+\(([^)]+)\)", r"\1", str(x))
)

# Rename columns to match formal Capstone rubric terminology
ibm_performance_df = ibm_performance_df.rename(columns={
    'specificity': 'Retention Recall (Class 0)',
    'sensitivity': 'Attrition Recall (Class 1)',
    'f1 score': 'F1-Score (Weighted)'
})

# Sort and rank the models by Attrition Recall (Our primary optimization metric)
ibm_performance_df['Rank'] = ibm_performance_df['Attrition Recall (Class 1)'].rank(ascending=False, method='min')
ibm_performance_df = ibm_performance_df.set_index('Rank').sort_index(ascending=True)

# Format to 3 decimal places for a clean, academic presentation
ibm_performance_df = ibm_performance_df.round(3)

# Force Pandas to wrap text in the PDF instead of clipping it off the page

print("\n=== IBM Dataset: Consolidated Algorithm Performance ===")

# Convert the dataframe to a native Markdown table so Quarto automatically wraps the text
display(Markdown(ibm_performance_df.to_markdown()))


=== IBM Dataset: Consolidated Algorithm Performance ===


|   Rank | model name                          | tuned hyperparameters                                                                     |   Retention Recall (Class 0) |   Attrition Recall (Class 1) |   F1-Score (Weighted) |
|-------:|:------------------------------------|:------------------------------------------------------------------------------------------|-----------------------------:|-----------------------------:|----------------------:|
|      1 | D: Decision Tree                    | {'min_samples_split': 2, 'min_samples_leaf': 20, 'max_depth': 5, 'criterion': 'log_loss'} |                        0.681 |                        0.75  |                 0.731 |
|      2 | F: Generalized Additive Model (GAM) | {'lam': 0.1}                                                                              |                        0.795 |                        0.722 |                 0.804 |
|      3 | C: Linear SVM (Tuned)               | {'C': 0.0379269019073225}                                                                 |                        0.741 |                        0.694 |                 0.764 |
|      3 | B: Regularized Logistic Regression  | {'solver': 'saga', 'penalty': 'l2', 'C': 555.556}                                         |                        0.735 |                        0.694 |                 0.76  |
|      5 | E: k-Nearest Neighbors              | n_neighbors=7                                                                             |                        0.957 |                        0.417 |                 0.857 |
|      6 | A: Standard Logistic Regression     | Baseline (Untuned)                                                                        |                        0.989 |                        0.222 |                 0.83  |

In [21]:
# ==========================================
# PART 2: THE BABUSHKIN DATASET PIPELINE
# ==========================================
# 1. Ingest the Raw Data
# (Ensure 'babushkin.csv' is in your working directory)
bab_df = pd.read_csv("babushkin.csv", encoding="cp1251")


# 2. Define the MARS Features (per Week 7 Mapping)
bab_mars_columns = [
    'event',         # Target Variable (Attrition Flag)
    'stag',          # Ability: Length of Service (Months)
    'greywage',      # Situational: Compensation Method
    'way',           # Situational: Commute Method
    'traffic',       # Situational: Source of Hire
    'industry',      # Role Perception: Industry Boundary
    'profession',    # Role Perception: Profession Boundary
    'coach',         # Role Perception: Mentorship Proxy
    'extraversion',  # Motivation: FFM Extraversion
    'anxiety',       # Motivation: FFM Neuroticism
    'selfcontrol',   # Motivation: FFM Conscientiousness
    'novator',       # Motivation: FFM Openness to Experience
    'independ'       # Motivation: FFM Agreeableness (Reverse Scale)
]

# 3. Drop Useless Features by Filtering
bab_df = bab_df[bab_mars_columns]

In [22]:
# --- 1. Binary Transformation: Greywage ---
# if 'grey' (tax-evading), 0 if 'white' (fully compliant) {#sec-}
bab_df['greywage'] = (bab_df['greywage'] == 'grey').astype(int)

# --- 2. Binary Transformation: Profession (HR Flag) ---
# if HR, 0 if any other profession {#sec-}
bab_df['hr_profession'] = (bab_df['profession'] == 'HR').astype(int)

# Drop the original 'profession' column since we've extracted the core signal
bab_df = bab_df.drop('profession', axis=1)

# --- 3. One-Hot Encoding: Coach and Way ---
# Convert the nominal categories into distinct binary columns
bab_df = pd.get_dummies(bab_df, columns=['coach', 'way'], drop_first=False, dtype=int)

In [23]:
# --- 4. Grouping & Encoding: Traffic and Industry ---

# Define the mapping dictionaries
traffic_mapping = {
    'youjs': 'Job_Boards', 'empjs': 'Job_Boards',
    'friends': 'Referral_Network', 'referal': 'Referral_Network', 
    'rabrecNErab': 'Referral_Network', 'recNErab': 'Referral_Network',
    'KA': 'Agency_Ads', 'advert': 'Agency_Ads'
}

industry_mapping = {
    'Retail': 'Retail_Services', 'HoReCa': 'Retail_Services', 'RealEstate': 'Retail_Services', 'etc': 'Retail_Services',
    'manufacture': 'Heavy_Industry', 'Building': 'Heavy_Industry', 'PowerGeneration': 'Heavy_Industry', 
    'Mining': 'Heavy_Industry', 'Agriculture': 'Heavy_Industry', 'transport': 'Heavy_Industry', 'Pharma': 'Heavy_Industry',
    'IT': 'Tech_Professional', 'Banks': 'Tech_Professional', 'Consult': 'Tech_Professional', 'Telecom': 'Tech_Professional',
    'State': 'Public_Sector'
}

# Apply the mappings to the columns
bab_df['traffic'] = bab_df['traffic'].map(traffic_mapping)
bab_df['industry'] = bab_df['industry'].map(industry_mapping)

# One-Hot Encode the newly grouped columns
bab_df = pd.get_dummies(bab_df, columns=['traffic', 'industry'], drop_first=False, dtype=int)

In [24]:
# ==========================================
# PART 3: BABUSHKIN MARS TAGGING & PREPROCESSING
# ==========================================
# 1. Separate Features (X) and Target (y)
bab_y = bab_df['event']
bab_X_vals = bab_df.drop('event', axis=1)

# 2. Define the MARS Tag Map for Babushkin
# Based on Week 7 text (lost-in-translation / Russian HR metrics)
bab_mars_tag_map = {
    # Ability
    'stag': '(A)', 
    
    # Situational
    'greywage': '(S)', 'way': '(S)', 'traffic': '(S)',
    
    # Role Perception
    'hr_profession': '(R)', 'industry': '(R)', 'coach': '(R)',
    
    # Motivation (Five Factor Model)
    'extraversion': '(M)', 'anxiety': '(M)', 'selfcontrol': '(M)', 
    'novator': '(M)', 'independ': '(M)'
}

# 3. Append MARS Tags to Feature Names
bab_X_labels = []
for col in bab_X_vals.columns:
    tag = ''
    for base_feature, mars_tag in bab_mars_tag_map.items():
        # Using startswith() perfectly catches our new one-hot encoded columns 
        # (e.g., 'traffic_Job_Boards' will successfully catch the 'traffic' base tag)
        if col.startswith(base_feature):
            tag = mars_tag
            break
    bab_X_labels.append(f"{col} {tag}".strip())

# Assign the updated labels back to the dataframe and extract values
bab_X_vals.columns = bab_X_labels
bab_X = bab_X_vals.values

# 4. Target Encoding
# Encoding target as integers to avoid model dtype warnings later
le = LabelEncoder()
bab_y = le.fit_transform(bab_y)

# 5. Feature Scaling
# MinMax scaling needed since SVM and kNN rely on distance metrics
scaling_obj = MinMaxScaler()
bab_X_sc = scaling_obj.fit_transform(bab_X)

# 6. Train/Test Splitting
# Create unscaled train/test splits (Used for Decision Trees)
bab_X_train, bab_X_test, bab_y_train, bab_y_test = train_test_split(
    bab_X, bab_y, test_size=0.15, random_state=16, stratify=bab_y
)

# Create scaled train/test splits (Used for Linear Models, SVM, and kNN)
bab_X_train_sc, bab_X_test_sc = train_test_split(
    bab_X_sc, bab_y, test_size=0.15, random_state=16, stratify=bab_y
)[:2]

In [25]:
# --- CLEAN PRE-PROCESSING CONSOLE LOG (BABUSHKIN) ---

# Compute raw weights for the Babushkin dataset
bab_raw_weights = compute_class_weight(class_weight='balanced', classes=np.unique(bab_y), y=bab_y)

# Converting numpy floats to native python types so it prints cleanly
bab_weights_dict = {int(cls): round(float(weight), 2) for cls, weight in zip(np.unique(bab_y), bab_raw_weights)}

# We hardcode the original shape here in case the raw dataframe variable was overwritten during filtering
bab_original_shape = (1129, 16)

# Single-line consolidated print statement (with the \n for PDF wrapping)
print(f"Babushkin Dataset Pipeline | Shape Filtered: {bab_original_shape} -> {bab_df.shape}\nComputed Imbalance Weights: {bab_weights_dict}")

print("\nFinal MARS-Tagged Features:")
for label in bab_X_labels:
    print(f" - {label}")

Babushkin Dataset Pipeline | Shape Filtered: (1129, 16) -> (1129, 22)
Computed Imbalance Weights: {0: 1.01, 1: 0.99}

Final MARS-Tagged Features:
 - stag (A)
 - greywage (S)
 - extraversion (M)
 - anxiety (M)
 - selfcontrol (M)
 - novator (M)
 - independ (M)
 - hr_profession (R)
 - coach_my head (R)
 - coach_no (R)
 - coach_yes (R)
 - way_bus (S)
 - way_car (S)
 - way_foot (S)
 - traffic_Agency_Ads (S)
 - traffic_Job_Boards (S)
 - traffic_Referral_Network (S)
 - industry_Heavy_Industry (R)
 - industry_Public_Sector (R)
 - industry_Retail_Services (R)
 - industry_Tech_Professional (R)


In [26]:
# --- RESET GLOBAL DICTIONARY FOR BABUSHKIN ---
results_dict = {
    'model name': [],
    'tuned hyperparameters': [],
    'specificity': [],
    'sensitivity': [],
    'f1 score': []
}

bab_lr_classifier = LogisticRegression(random_state=12)
bab_lr_classifier.fit(bab_X_train_sc, bab_y_train)

bab_y_pred_lr = bab_lr_classifier.predict(bab_X_test_sc)

# show_visuals explicitly set to False to prevent the confusion matrix from printing
classification_summary(bab_y_test, bab_y_pred_lr, 'A: Standard Logistic Regression', show_visuals=False)

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    
    # --- Model B: Regularized Logistic Regression ---
    bab_logreg_params = {
        "C": np.linspace(0.001, 1000, 10),
        "solver": ['newton-cg', 'saga'],
        "penalty": ['l2'],
        "class_weight": [bab_weights_dict, None]
    }
    bab_logreg = LogisticRegression(max_iter=10000)

    bab_logreg_grid = RandomizedSearchCV(bab_logreg, bab_logreg_params, random_state=6, n_iter=20, scoring='f1')
    bab_logreg_grid.fit(bab_X_train_sc, bab_y_train)

    bab_l2logreg_model = LogisticRegression(**bab_logreg_grid.best_params_)
    bab_l2logreg_model.fit(bab_X_train_sc, bab_y_train)
    bab_y_pred_l2_logreg = bab_l2logreg_model.predict(bab_X_test_sc)

    classification_summary(
        bab_y_test, 
        bab_y_pred_l2_logreg, 
        'B: Regularized Logistic Regression', 
        tuned_params=bab_logreg_grid.best_params_,
        show_visuals=False
    )

    # --- Model C: Linear SVM ---
    bab_svm_params = {
        "C": np.logspace(-3, 3, 20),
        "class_weight": [bab_weights_dict]
    }
    bab_svm = LinearSVC(penalty='l2', loss='squared_hinge', dual=False, max_iter=20000)

    bab_svm_grid = RandomizedSearchCV(bab_svm, bab_svm_params, random_state=21, scoring='f1')
    bab_svm_grid.fit(bab_X_train_sc, bab_y_train)

    bab_best_svm = bab_svm_grid.best_estimator_
    bab_y_pred_svm = bab_best_svm.predict(bab_X_test_sc)

    classification_summary(
        bab_y_test, 
        bab_y_pred_svm, 
        'C: Linear SVM (Tuned)', 
        tuned_params=bab_svm_grid.best_params_,
        show_visuals=False
    )

    # --- Model D: Decision Tree ---
    bab_tree_params = {
        "max_depth": np.arange(3, 10, 2),
        "min_samples_split": np.arange(2, 9, 1),
        "min_samples_leaf": np.arange(20, 50, 10),
        "criterion": ['gini', 'log_loss'],
        "class_weight": [bab_weights_dict, None]
    }
    bab_tree = DecisionTreeClassifier(random_state=12)

    bab_tree_grid = RandomizedSearchCV(bab_tree, bab_tree_params, random_state=6, n_iter=50, scoring='f1')
    # Note: Decision Trees use the unscaled features (bab_X_train)
    bab_tree_grid.fit(bab_X_train, bab_y_train)

    bab_tree_model = DecisionTreeClassifier(**bab_tree_grid.best_params_, random_state=12)
    bab_tree_model.fit(bab_X_train, bab_y_train)

    bab_y_pred_dt = bab_tree_model.predict(bab_X_test)

    classification_summary(
        bab_y_test, 
        bab_y_pred_dt, 
        'D: Decision Tree', 
        tuned_params=bab_tree_grid.best_params_,
        show_visuals=False
    )

In [27]:
# --- Model F: Generalized Additive Model (GAM) [BABUSHKIN OPTIMIZED] ---
# Map the computed class weights to every individual sample in the training set
bab_gam_sample_weights = np.array([bab_weights_dict[cls] for cls in bab_y_train])

# OPTIMIZATION: Shrink the search space to 5 lambdas to drastically speed up render time
bab_gam_lams = np.logspace(-2, 2, 5)

# Dropping n_splines to 10 to speed up execution and prevent overfitting the noise
bab_gam_model = LogisticGAM(n_splines=10)

# Execute gridsearch with sample weights applied, progress bar suppressed, and math warnings silenced
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    bab_gam_model.gridsearch(
        bab_X_train_sc, 
        bab_y_train, 
        lam=bab_gam_lams, 
        weights=bab_gam_sample_weights, 
        progress=False
    )

bab_y_pred_gam = bab_gam_model.predict(bab_X_test_sc)

classification_summary(
    bab_y_test, 
    bab_y_pred_gam, 
    'F: Generalized Additive Model (GAM)', 
    tuned_params={'lam': round(float(bab_gam_model.lam[0][0]), 3)},
    show_visuals=False
)

In [28]:
#| label: fig-bab-knn
#| fig-cap: 'KNN Elbow Diagram: Attrition Recall vs. Neighbors (Babushkin Dataset)'

# Initialize variables for the Elbow Diagram...

# We use f_classif which handles the strictly positive values from MinMaxScaler
bab_selector = SelectKBest(score_func=f_classif, k=10)
bab_X_train_selected = bab_selector.fit_transform(bab_X_train_sc, bab_y_train)
bab_X_test_selected = bab_selector.transform(bab_X_test_sc)

bab_selected_mask = bab_selector.get_support()
bab_selected_features = np.array(bab_X_labels)[bab_selected_mask]

print(f"kNN will now use only these {len(bab_selected_features)} features:")
print(bab_selected_features)

# Initialize variables for the Elbow Diagram
bab_neighbors = range(1, 20)
bab_knn_train_recall = {}
bab_knn_test_recall = {}

for neighbor in bab_neighbors:
    # Initialize and Fit 
    # Added weights='distance' to prevent the majority class from swamping the vote
    bab_knn = KNeighborsClassifier(n_neighbors=neighbor, weights='distance')
    bab_knn.fit(bab_X_train_selected, bab_y_train)
    
    # Predict
    bab_y_pred_train_knn = bab_knn.predict(bab_X_train_selected)
    bab_y_pred_test_knn = bab_knn.predict(bab_X_test_selected)
    
    # Calculate RECALL for the Positive Class (Attrition)
    bab_knn_train_recall[neighbor] = recall_score(bab_y_train, bab_y_pred_train_knn)
    bab_knn_test_recall[neighbor] = recall_score(bab_y_test, bab_y_pred_test_knn)

# Plotting the Elbow Diagram anchored on Recall
plt.figure(figsize=(10, 6))
plt.title("KNN Elbow Diagram: Attrition Recall vs Neighbors (Babushkin Dataset)")
plt.plot(bab_neighbors, list(bab_knn_train_recall.values()), label="Training Recall", marker='o')
plt.plot(bab_neighbors, list(bab_knn_test_recall.values()), label="Testing Recall", marker='o')
plt.legend()
plt.xlabel("Number of Neighbors (k)")
plt.ylabel("Recall (Sensitivity for Class 1: Attrition)")
plt.grid(True, linestyle='--', alpha=0.7)
plt.xticks(bab_neighbors) 
plt.show()

# OVERRIDE: Hardcode the optimal k for business interpretability (work pod size)
bab_best_k = 5
print(f"Manually Selected for Interpretability: k={bab_best_k} with Test Recall of {bab_knn_test_recall[bab_best_k]:.4f}")

# Train the finalized KNN model with the optimal k
bab_knn_optimized = KNeighborsClassifier(n_neighbors=bab_best_k, weights='distance')
bab_knn_optimized.fit(bab_X_train_selected, bab_y_train)
bab_y_pred_knn = bab_knn_optimized.predict(bab_X_test_selected)

# Pass the final results to the silent summary tracker
classification_summary(
    bab_y_test, 
    bab_y_pred_knn, 
    f'E: k-Nearest Neighbors', 
    tuned_params=f"n_neighbors={bab_best_k}",
    show_visuals=False
)

kNN will now use only these 10 features:
['stag (A)' 'anxiety (M)' 'independ (M)' 'hr_profession (R)'
 'coach_yes (R)' 'way_bus (S)' 'way_foot (S)' 'traffic_Job_Boards (S)'
 'traffic_Referral_Network (S)' 'industry_Public_Sector (R)']


<Figure size 3000x1800 with 1 Axes>

Manually Selected for Interpretability: k=5 with Test Recall of 0.7209


In [29]:
#| label: fig-bab-vims
#| fig-cap: Global Feature Importance by Algorithm (Babushkin Dataset)

# --- REFACTORED MASTER VISUAL FUNCTION ---

# Model A: Standard Logistic Regression (Directional)
store_feature_importances(bab_X_labels, bab_lr_classifier.coef_[0], 'A: Standard Logistic Regression', is_directional=True, dataset='Babushkin')

# Model B: Regularized Ridge (Directional)
store_feature_importances(bab_X_labels, bab_l2logreg_model.coef_[0], 'B: Regularized Logistic Regression', is_directional=True, dataset='Babushkin')

# Model C: Linear SVM (Directional)
store_feature_importances(bab_X_labels, bab_best_svm.coef_[0], 'C: Linear SVM (Tuned)', is_directional=True, dataset='Babushkin')

# Model D: Decision Tree (Magnitude Only)
store_feature_importances(bab_X_labels, bab_tree_model.feature_importances_, 'D: Decision Tree', is_directional=False, dataset='Babushkin')

# Model E: kNN (Magnitude Only via ANOVA F-Scores)
store_feature_importances(bab_X_labels, bab_selector.scores_, f'E: kNN (k={bab_best_k})', is_directional=False, dataset='Babushkin')

# Model F: Generalized Additive Model (Magnitude Only via Permutation Importance)
bab_gam_perm_imp = permutation_importance(bab_gam_model, bab_X_test_sc, bab_y_test, n_repeats=5, random_state=42)
store_feature_importances(bab_X_labels, bab_gam_perm_imp.importances_mean, 'F: Generalized Additive Model (GAM)', is_directional=False, dataset='Babushkin')

# --- EXECUTE THE BABUSHKIN PLOT ---
plot_vims_facet_grid(dataset='Babushkin')

<Figure size 4200x5400 with 6 Axes>

In [30]:
#| label: fig-bab-pdp
#| fig-cap: Non-Linear Feature Effects for Top Continuous Drivers (Babushkin GAM)

# --- MODEL F: PARTIAL DEPENDENCE PLOTS (BABUSHKIN) ---
# 1. Explicitly select the Top 3 CONTINUOUS features from the GAM VIMS
# Bypassing binary flags (like Retail_Services) because line-plot PDPs for discrete 0/1 data are visually misleading
bab_top_3_gam_features = ['anxiety (M)', 'novator (M)', 'selfcontrol (M)']

# Find their exact column indices in the scaled X matrix
bab_top_3_indices = [bab_X_labels.index(feat) for feat in bab_top_3_gam_features]

# 2. Create the 3x1 Grid
fig, axes = plt.subplots(nrows=3, ncols=1, figsize=(10, 15))

for i, (feat, idx) in enumerate(zip(bab_top_3_gam_features, bab_top_3_indices)):
    ax = axes[i]
    
    # Generate the grid for the specific term
    XX = bab_gam_model.generate_X_grid(term=idx)
    
    # Calculate dependence
    pdp = bab_gam_model.partial_dependence(term=idx, X=XX)
    
    # Calculate confidence intervals
    confi = bab_gam_model.confidence_intervals(XX, width=0.95)
    
    # Plotting the curve and the confidence bounds
    ax.plot(XX[:, idx], pdp, color='purple', linewidth=2)
    ax.fill_between(XX[:, idx], confi[:, 0], confi[:, 1], color='purple', alpha=0.2)
    
    # Baseline
    ax.axhline(0, color='black', linestyle='--', linewidth=1.1)
    
    # Formatting
    ax.set_title(f"PDP: {feat}", fontsize=12, fontweight='bold')
    ax.set_xlabel("Scaled Value (0 to 1)")
    ax.set_ylabel("Partial Dependence (Log-Odds)")
    
    ax.grid(True, linestyle=':', alpha=0.6)

fig.suptitle('Non-Linear Feature Effects: Top Continuous Drivers (Babushkin GAM)', fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

<Figure size 3000x4500 with 3 Axes>

In [31]:
get_top_kill_zones(bab_tree_model, bab_X_labels, ['Stay', 'Exit'])

=== Top 3 Attrition Kill Zones Found ===

Kill Zone 1: 63.6% Attrition Concentration
Impact: 0.6/1.0 weighted score (Applies to 261 actual employees | 27.22% of workforce)
Logic Path: stag (A) > 2.73 AND stag (A) <= 27.09 AND independ (M) > 5.15

Kill Zone 2: 55.1% Attrition Concentration
Impact: 0.6/1.0 weighted score (Applies to 256 actual employees | 26.69% of workforce)
Logic Path: stag (A) > 2.73 AND stag (A) > 27.09 AND traffic_Job_Boards (S) <= 0.50

Kill Zone 3: 48.7% Attrition Concentration
Impact: 0.5/1.0 weighted score (Applies to 193 actual employees | 20.13% of workforce)
Logic Path: stag (A) > 2.73 AND stag (A) <= 27.09 AND independ (M) <= 5.15


In [32]:
# --- PREP DATASET FOR SHAP (BABUSHKIN) ---
# Re-attaching labels for readability in the waterfall plots
bab_X_test_df = pd.DataFrame(bab_X_test_sc, columns=bab_X_labels)
bab_X_train_df = pd.DataFrame(bab_X_train_sc, columns=bab_X_labels)

# --- 1. MODEL A: Standard Logistic Regression ---
bab_probas_a = bab_lr_classifier.predict_proba(bab_X_test_sc)[:, 1]
bab_idx_a_min, bab_idx_a_max = np.argmin(bab_probas_a), np.argmax(bab_probas_a)
bab_explainer_a = shap.LinearExplainer(bab_lr_classifier, bab_X_train_df)
bab_shap_a = bab_explainer_a(bab_X_test_df)

# --- 2. MODEL B: Regularized Logistic Regression ---
bab_probas_b = bab_l2logreg_model.predict_proba(bab_X_test_sc)[:, 1]
bab_idx_b_min, bab_idx_b_max = np.argmin(bab_probas_b), np.argmax(bab_probas_b)
bab_explainer_b = shap.LinearExplainer(bab_l2logreg_model, bab_X_train_df)
bab_shap_b = bab_explainer_b(bab_X_test_df)

# --- 3. MODEL C: Linear SVM ---
bab_scores_c = bab_best_svm.decision_function(bab_X_test_sc)
bab_idx_c_min, bab_idx_c_max = np.argmin(bab_scores_c), np.argmax(bab_scores_c)
bab_explainer_c = shap.LinearExplainer(bab_best_svm, bab_X_train_df)
bab_shap_c = bab_explainer_c(bab_X_test_df)

# --- 4. MODEL F: Generalized Additive Model (GAM) ---
bab_probas_f = bab_gam_model.predict_proba(bab_X_test_sc)
bab_idx_f_min, bab_idx_f_max = np.argmin(bab_probas_f), np.argmax(bab_probas_f)

# We use a smaller background sample for GAM SHAP to speed up the kernel calculation
bab_gam_bg = shap.sample(bab_X_train_df, 50) 
bab_explainer_f = shap.Explainer(bab_gam_model.predict, bab_gam_bg)

# Slicing just the min/max rows here, otherwise KernelExplainer takes forever to run
bab_shap_f_safe = bab_explainer_f(bab_X_test_df.iloc[[bab_idx_f_min]], silent=True)
bab_shap_f_risk = bab_explainer_f(bab_X_test_df.iloc[[bab_idx_f_max]], silent=True)

In [33]:
#| layout-ncol: 2
#| label: fig-bab-shap-safe
#| fig-cap: 'SHAP Local Interpretability: Profiles with Lowest Attrition Probability (Babushkin)'
#| fig-subcap:
#|   - 'A: Std Logistic Regression - Safest'
#|   - 'B: Reg. Logistic Regression - Safest'
#|   - 'C: Linear SVM (Tuned) - Safest'
#|   - 'F: GAM - Safest'

# --- THE "SAFE" ANCHORS (Lowest Probabilities for Babushkin) ---
# Calling the render_shap_waterfall function defined in the IBM section

render_shap_waterfall(bab_shap_a[bab_idx_a_min], "")
render_shap_waterfall(bab_shap_b[bab_idx_b_min], "")
render_shap_waterfall(bab_shap_c[bab_idx_c_min], "")
render_shap_waterfall(bab_shap_f_safe[0], "")

<Figure size 2400x2400 with 3 Axes>

<Figure size 2400x2400 with 3 Axes>

<Figure size 2400x2400 with 3 Axes>

<Figure size 2400x2400 with 3 Axes>

In [34]:
#| layout-ncol: 2
#| label: fig-bab-shap-risk
#| fig-cap: 'SHAP Local Interpretability: Profiles with Highest Attrition Probability (Babushkin)'
#| fig-subcap:
#|   - 'A: Std Logistic Regression - Riskiest'
#|   - 'B: Reg. Logistic Regression - Riskiest'
#|   - 'C: Linear SVM (Tuned) - Riskiest'
#|   - 'F: GAM - Riskiest'



# --- THE "FLIGHT RISKS" (Highest Probabilities for Babushkin) ---

render_shap_waterfall(bab_shap_a[bab_idx_a_max], "")
render_shap_waterfall(bab_shap_b[bab_idx_b_max], "")
render_shap_waterfall(bab_shap_c[bab_idx_c_max], "")
render_shap_waterfall(bab_shap_f_risk[0], "")

<Figure size 2400x2400 with 3 Axes>

<Figure size 2400x2400 with 3 Axes>

<Figure size 2400x2400 with 3 Axes>

<Figure size 2400x2400 with 3 Axes>

In [35]:
#| label: tbl-bab-perf
#| tbl-cap: Consolidated Algorithm Performance and Hyperparameters (Babushkin Dataset)

# 4. Universal Performance Matrix (Babushkin Dataset)
# Convert the populated results dictionary into a master dataframe
bab_performance_df = pd.DataFrame(results_dict)

# Using regex to strip out numpy wrappers from the hyperparameter strings so the table looks clean
bab_performance_df['tuned hyperparameters'] = bab_performance_df['tuned hyperparameters'].apply(
    lambda x: re.sub(r"np\.\w+\(([^)]+)\)", r"\1", str(x))
)

# Rename columns to match formal Capstone rubric terminology
bab_performance_df = bab_performance_df.rename(columns={
    'specificity': 'Retention Recall (Class 0)',
    'sensitivity': 'Attrition Recall (Class 1)',
    'f1 score': 'F1-Score (Weighted)'
})

# Sort and rank the models by Attrition Recall (Our primary optimization metric)
bab_performance_df['Rank'] = bab_performance_df['Attrition Recall (Class 1)'].rank(ascending=False, method='min')
bab_performance_df = bab_performance_df.set_index('Rank').sort_index(ascending=True)

# Format to 3 decimal places for a clean, academic presentation
bab_performance_df = bab_performance_df.round(3)

print("\n=== Babushkin Dataset: Consolidated Algorithm Performance ===")

# Convert the dataframe to a native Markdown table so Quarto automatically wraps the text
display(Markdown(bab_performance_df.to_markdown()))


=== Babushkin Dataset: Consolidated Algorithm Performance ===


|   Rank | model name                          | tuned hyperparameters                                                                     |   Retention Recall (Class 0) |   Attrition Recall (Class 1) |   F1-Score (Weighted) |
|-------:|:------------------------------------|:------------------------------------------------------------------------------------------|-----------------------------:|-----------------------------:|----------------------:|
|      1 | B: Regularized Logistic Regression  | {'solver': 'saga', 'penalty': 'l2', 'C': 0.001}                                           |                        0.071 |                        0.953 |                 0.4   |
|      2 | E: k-Nearest Neighbors              | n_neighbors=5                                                                             |                        0.583 |                        0.721 |                 0.651 |
|      3 | F: Generalized Additive Model (GAM) | {'lam': 100.0}                                                                            |                        0.56  |                        0.651 |                 0.605 |
|      3 | A: Standard Logistic Regression     | Baseline (Untuned)                                                                        |                        0.56  |                        0.651 |                 0.605 |
|      5 | C: Linear SVM (Tuned)               | {'C': 0.0379269019073225}                                                                 |                        0.583 |                        0.616 |                 0.6   |
|      6 | D: Decision Tree                    | {'min_samples_split': 6, 'min_samples_leaf': 20, 'max_depth': 3, 'criterion': 'log_loss'} |                        0.476 |                        0.558 |                 0.517 |

In [36]:
# ==========================================
# PART 3: THE DATA SCIENTIST JOB CHANGE PIPELINE
# ==========================================

# 1. Ingest the Raw Data
# (Ensure 'dsjobchange.csv' is in your working directory)
ds_df = pd.read_csv("dsjobchange.csv")

# 2. Define the MARS Features (per Week 10 Text)
ds_mars_columns = [
    'target',                  # Target Variable (Attrition Flag)
    'enrolled_university',     # Motivation: Education pursuit
    'last_new_job',            # Motivation: Job-hopping velocity
    'relevent_experience',     # Ability: Domain expertise
    'experience',              # Ability: Total longitudinal experience
    'company_size',            # Role Perception: Structural boundary
    'company_type',            # Role Perception: Enterprise type
    'training_hours',          # Role Perception: Formal onboarding
    'city_development_index'   # Situational: Macro-environmental stressor
]

# 3. Drop Useless Features by Filtering
ds_df = ds_df[ds_mars_columns]

In [37]:
# Fix 'experience' column: '<1' to 0, '>20' to 21, then cast to float
ds_df['experience'] = ds_df['experience'].replace({'<1': 0, '>20': 21})
ds_df['experience'] = pd.to_numeric(ds_df['experience'], errors='coerce')

# Fix 'last_new_job' column: 'never' to 0, '>4' to 5, then cast to float
ds_df['last_new_job'] = ds_df['last_new_job'].replace({'never': 0, '>4': 5})
ds_df['last_new_job'] = pd.to_numeric(ds_df['last_new_job'], errors='coerce')

# Impute any resulting NaNs in our new numerical columns with the median
ds_df['experience'] = ds_df['experience'].fillna(ds_df['experience'].median())
ds_df['last_new_job'] = ds_df['last_new_job'].fillna(ds_df['last_new_job'].median())

# --- CATEGORICAL TRANSFORMATIONS ---

# Fix the Excel date artifact in company_size
ds_df['company_size'] = ds_df['company_size'].replace({'54697': '10-100'})
# (Adding a safeguard for the other common Kaggle/Excel glitch just in case)
ds_df['company_size'] = ds_df['company_size'].replace({'10/49': '10-49'}) 

# Define the remaining purely categorical columns
ds_cat_cols = [
    'enrolled_university', 'relevent_experience', 
    'company_size', 'company_type'
]

# Apply the Constant Imputation strategy to the text columns
ds_df[ds_cat_cols] = ds_df[ds_cat_cols].fillna('Unknown')

# One-Hot Encode the string categories into binary sparse matrices
ds_df = pd.get_dummies(ds_df, columns=ds_cat_cols, drop_first=False, dtype=int)

In [38]:
# ==========================================
# PART 4: DS JOB CHANGE MARS TAGGING & PREPROCESSING
# ==========================================


# 1. Separate Features (X) and Target (y)
ds_y = ds_df['target']
ds_X_vals = ds_df.drop('target', axis=1)

# 2. Define the MARS Tag Map for Data Scientist Dataset
# Based on Week 10 text justifications
ds_mars_tag_map = {
    # Motivation (Intrinsic upskilling & job velocity)
    'enrolled_university': '(M)', 'last_new_job': '(M)',
    
    # Ability (Domain expertise & longitudinal history)
    'relevent_experience': '(A)', 'experience': '(A)',
    
    # Role Perception (Structural boundaries & onboarding)
    'company_size': '(R)', 'company_type': '(R)', 'training_hours': '(R)',
    
    # Situational (Macro-environmental stressor)
    'city_development_index': '(S)'
}

# 3. Append MARS Tags to Feature Names
ds_X_labels = []
for col in ds_X_vals.columns:
    tag = ''
    for base_feature, mars_tag in ds_mars_tag_map.items():
        # Using startswith() perfectly catches our new one-hot encoded columns 
        # (e.g., 'company_size_10-100' will successfully catch the 'company_size' base tag)
        if col.startswith(base_feature):
            tag = mars_tag
            break
    ds_X_labels.append(f"{col} {tag}".strip())

# Assign the updated labels back to the dataframe and extract values
ds_X_vals.columns = ds_X_labels
ds_X = ds_X_vals.values

# 4. Target Encoding
# Encoding target as integers to avoid model dtype warnings later
le = LabelEncoder()
ds_y = le.fit_transform(ds_y)

# 5. Feature Scaling
# MinMax scaling needed since SVM and kNN rely on distance metrics
scaling_obj = MinMaxScaler()
ds_X_sc = scaling_obj.fit_transform(ds_X)

# 6. Train/Test Splitting
# Create unscaled train/test splits (Used for Decision Trees)
ds_X_train, ds_X_test, ds_y_train, ds_y_test = train_test_split(
    ds_X, ds_y, test_size=0.15, random_state=16, stratify=ds_y
)

# Create scaled train/test splits (Used for Linear Models, SVM, and kNN)
ds_X_train_sc, ds_X_test_sc = train_test_split(
    ds_X_sc, ds_y, test_size=0.15, random_state=16, stratify=ds_y
)[:2]

In [39]:
# --- CLEAN PRE-PROCESSING CONSOLE LOG (DS JOB CHANGE) ---

# Compute raw weights for the DS dataset
ds_raw_weights = compute_class_weight(class_weight='balanced', classes=np.unique(ds_y), y=ds_y)

# Converting numpy floats to native python types so it prints cleanly
ds_weights_dict = {int(cls): round(float(weight), 2) for cls, weight in zip(np.unique(ds_y), ds_raw_weights)}

# We hardcode the original shape here in case the raw dataframe variable was overwritten during filtering
ds_original_shape = (19158, 14)

# Single-line consolidated print statement (with the \n for PDF wrapping)
print(f"DS Job Change Dataset Pipeline | Shape Filtered: {ds_original_shape} -> {ds_df.shape}\nComputed Imbalance Weights: {ds_weights_dict}")

print("\nFinal MARS-Tagged Features:")
for label in ds_X_labels:
    print(f" - {label}")

DS Job Change Dataset Pipeline | Shape Filtered: (19158, 14) -> (19158, 27)
Computed Imbalance Weights: {0: 0.67, 1: 2.01}

Final MARS-Tagged Features:
 - last_new_job (M)
 - experience (A)
 - training_hours (R)
 - city_development_index (S)
 - enrolled_university_Full time course (M)
 - enrolled_university_Part time course (M)
 - enrolled_university_Unknown (M)
 - enrolled_university_no_enrollment (M)
 - relevent_experience_Has relevent experience (A)
 - relevent_experience_No relevent experience (A)
 - company_size_10-100 (R)
 - company_size_100-500 (R)
 - company_size_1000-4999 (R)
 - company_size_10000+ (R)
 - company_size_50-99 (R)
 - company_size_500-999 (R)
 - company_size_5000-9999 (R)
 - company_size_<10 (R)
 - company_size_Unknown (R)
 - company_type_Early Stage Startup (R)
 - company_type_Funded Startup (R)
 - company_type_NGO (R)
 - company_type_Other (R)
 - company_type_Public Sector (R)
 - company_type_Pvt Ltd (R)
 - company_type_Unknown (R)


In [40]:
# --- RESET GLOBAL DICTIONARY FOR DS JOB CHANGE ---
results_dict = {
    'model name': [],
    'tuned hyperparameters': [],
    'specificity': [],
    'sensitivity': [],
    'f1 score': []
}

# --- Model A: Standard Logistic Regression ---
ds_lr_classifier = LogisticRegression(random_state=12)
ds_lr_classifier.fit(ds_X_train_sc, ds_y_train)

ds_y_pred_lr = ds_lr_classifier.predict(ds_X_test_sc)

# show_visuals explicitly set to False to prevent the confusion matrix from printing
classification_summary(ds_y_test, ds_y_pred_lr, 'A: Standard Logistic Regression', show_visuals=False)

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    
    # --- Model B: Regularized Logistic Regression ---
    ds_logreg_params = {
        "C": np.linspace(0.001, 1000, 10),
        "solver": ['newton-cg', 'saga'],
        "penalty": ['l2'],
        "class_weight": [ds_weights_dict, None]
    }
    ds_logreg = LogisticRegression(max_iter=10000)

    ds_logreg_grid = RandomizedSearchCV(ds_logreg, ds_logreg_params, random_state=6, n_iter=20, scoring='f1')
    ds_logreg_grid.fit(ds_X_train_sc, ds_y_train)

    ds_l2logreg_model = LogisticRegression(**ds_logreg_grid.best_params_)
    ds_l2logreg_model.fit(ds_X_train_sc, ds_y_train)
    ds_y_pred_l2_logreg = ds_l2logreg_model.predict(ds_X_test_sc)

    classification_summary(
        ds_y_test, 
        ds_y_pred_l2_logreg, 
        'B: Regularized Logistic Regression', 
        tuned_params=ds_logreg_grid.best_params_,
        show_visuals=False
    )

    # --- Model C: Linear SVM ---
    ds_svm_params = {
        "C": np.logspace(-3, 3, 20),
        "class_weight": [ds_weights_dict]
    }
    ds_svm = LinearSVC(penalty='l2', loss='squared_hinge', dual=False, max_iter=20000)

    ds_svm_grid = RandomizedSearchCV(ds_svm, ds_svm_params, random_state=21, scoring='f1')
    ds_svm_grid.fit(ds_X_train_sc, ds_y_train)

    ds_best_svm = ds_svm_grid.best_estimator_
    ds_y_pred_svm = ds_best_svm.predict(ds_X_test_sc)

    classification_summary(
        ds_y_test, 
        ds_y_pred_svm, 
        'C: Linear SVM (Tuned)', 
        tuned_params=ds_svm_grid.best_params_,
        show_visuals=False
    )

    # --- Model D: Decision Tree ---
    ds_tree_params = {
        "max_depth": np.arange(3, 10, 2),
        "min_samples_split": np.arange(2, 9, 1),
        "min_samples_leaf": np.arange(20, 50, 10),
        "criterion": ['gini', 'log_loss'],
        "class_weight": [ds_weights_dict, None]
    }
    ds_tree = DecisionTreeClassifier(random_state=12)

    ds_tree_grid = RandomizedSearchCV(ds_tree, ds_tree_params, random_state=6, n_iter=50, scoring='f1')
    # Note: Decision Trees use the unscaled features (ds_X_train)
    ds_tree_grid.fit(ds_X_train, ds_y_train)

    ds_tree_model = DecisionTreeClassifier(**ds_tree_grid.best_params_, random_state=12)
    ds_tree_model.fit(ds_X_train, ds_y_train)

    ds_y_pred_dt = ds_tree_model.predict(ds_X_test)

    classification_summary(
        ds_y_test, 
        ds_y_pred_dt, 
        'D: Decision Tree', 
        tuned_params=ds_tree_grid.best_params_,
        show_visuals=False
    )

In [41]:
# --- Model F: Generalized Additive Model (GAM) [DS OPTIMIZED] ---
# Map the computed class weights to every individual sample in the training set
ds_gam_sample_weights = np.array([ds_weights_dict[cls] for cls in ds_y_train])

# OPTIMIZATION: Shrink the search space to 5 lambdas to drastically speed up render time
ds_gam_lams = np.logspace(-2, 2, 5)

# Dropping n_splines to 10 to speed up execution and prevent overfitting the noise
ds_gam_model = LogisticGAM(n_splines=10)

# Execute gridsearch with sample weights applied, progress bar suppressed, and math warnings silenced
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    ds_gam_model.gridsearch(
        ds_X_train_sc, 
        ds_y_train, 
        lam=ds_gam_lams, 
        weights=ds_gam_sample_weights, 
        progress=False
    )

ds_y_pred_gam = ds_gam_model.predict(ds_X_test_sc)

classification_summary(
    ds_y_test, 
    ds_y_pred_gam, 
    'F: Generalized Additive Model (GAM)', 
    tuned_params={'lam': round(float(ds_gam_model.lam[0][0]), 3)},
    show_visuals=False
)

did not converge


In [42]:
#| label: fig-ds-knn
#| fig-cap: 'KNN Elbow Diagram: Attrition Recall vs. Neighbors (Data Science Dataset)'

# Initialize variables for the Elbow Diagram...

# We use f_classif which handles the strictly positive values from MinMaxScaler
ds_selector = SelectKBest(score_func=f_classif, k=10)
ds_X_train_selected = ds_selector.fit_transform(ds_X_train_sc, ds_y_train)
ds_X_test_selected = ds_selector.transform(ds_X_test_sc)

ds_selected_mask = ds_selector.get_support()
ds_selected_features = np.array(ds_X_labels)[ds_selected_mask]

print(f"kNN will now use only these {len(ds_selected_features)} features:")
print(ds_selected_features)

# Initialize variables for the Elbow Diagram
ds_neighbors = range(1, 20)
ds_knn_train_recall = {}
ds_knn_test_recall = {}

for neighbor in ds_neighbors:
    # Initialize and Fit 
    # Added weights='distance' to prevent the majority class from swamping the vote
    ds_knn = KNeighborsClassifier(n_neighbors=neighbor, weights='distance')
    ds_knn.fit(ds_X_train_selected, ds_y_train)
    
    # Predict
    ds_y_pred_train_knn = ds_knn.predict(ds_X_train_selected)
    ds_y_pred_test_knn = ds_knn.predict(ds_X_test_selected)
    
    # Calculate RECALL for the Positive Class (Attrition)
    ds_knn_train_recall[neighbor] = recall_score(ds_y_train, ds_y_pred_train_knn)
    ds_knn_test_recall[neighbor] = recall_score(ds_y_test, ds_y_pred_test_knn)

# Plotting the Elbow Diagram anchored on Recall
plt.figure(figsize=(10, 6))
plt.title("KNN Elbow Diagram: Attrition Recall vs Neighbors (Data Scientist Dataset)")
plt.plot(ds_neighbors, list(ds_knn_train_recall.values()), label="Training Recall", marker='o')
plt.plot(ds_neighbors, list(ds_knn_test_recall.values()), label="Testing Recall", marker='o')
plt.legend()
plt.xlabel("Number of Neighbors (k)")
plt.ylabel("Recall (Sensitivity for Class 1: Attrition)")
plt.grid(True, linestyle='--', alpha=0.7)
plt.xticks(ds_neighbors) 
plt.show()

# Extract the exact best k dynamically based purely on max Test Recall
ds_best_k = max(ds_knn_test_recall, key=ds_knn_test_recall.get)
print(f"Peak Performance: k={ds_best_k} with Test Recall of {ds_knn_test_recall[ds_best_k]:.4f}")

# Train the finalized KNN model with the optimal k
ds_knn_optimized = KNeighborsClassifier(n_neighbors=ds_best_k, weights='distance')
ds_knn_optimized.fit(ds_X_train_selected, ds_y_train)
ds_y_pred_knn = ds_knn_optimized.predict(ds_X_test_selected)

# Pass the final results to the silent summary tracker
classification_summary(
    ds_y_test, 
    ds_y_pred_knn, 
    f'E: k-Nearest Neighbors', 
    tuned_params=f"n_neighbors={ds_best_k}",
    show_visuals=False
)

kNN will now use only these 10 features:
['last_new_job (M)' 'experience (A)' 'city_development_index (S)'
 'enrolled_university_Full time course (M)'
 'enrolled_university_no_enrollment (M)'
 'relevent_experience_Has relevent experience (A)'
 'relevent_experience_No relevent experience (A)'
 'company_size_Unknown (R)' 'company_type_Pvt Ltd (R)'
 'company_type_Unknown (R)']


<Figure size 3000x1800 with 1 Axes>

Peak Performance: k=3 with Test Recall of 0.4644


In [43]:
#| label: fig-ds-vims
#| fig-cap: Global Feature Importance by Algorithm (Data Science Dataset)

# --- POPULATE THE DS JOB CHANGE VIMS DATAFRAME ---
from sklearn.inspection import permutation_importance

# Model A: Standard Logistic Regression (Directional)
store_feature_importances(ds_X_labels, ds_lr_classifier.coef_[0], 'A: Standard Logistic Regression', is_directional=True, dataset='DS_Job')

# Model B: Regularized Ridge (Directional)
store_feature_importances(ds_X_labels, ds_l2logreg_model.coef_[0], 'B: Regularized Logistic Regression', is_directional=True, dataset='DS_Job')

# Model C: Linear SVM (Directional)
store_feature_importances(ds_X_labels, ds_best_svm.coef_[0], 'C: Linear SVM (Tuned)', is_directional=True, dataset='DS_Job')

# Model D: Decision Tree (Magnitude Only)
store_feature_importances(ds_X_labels, ds_tree_model.feature_importances_, 'D: Decision Tree', is_directional=False, dataset='DS_Job')

# Model E: kNN (Magnitude Only via ANOVA F-Scores)
store_feature_importances(ds_X_labels, ds_selector.scores_, f'E: kNN (k={ds_best_k})', is_directional=False, dataset='DS_Job')

# Model F: Generalized Additive Model (Magnitude Only via Permutation Importance)
# Note: Permutation importance on 19k rows takes a moment, but is much faster than the initial gridsearch
ds_gam_perm_imp = permutation_importance(ds_gam_model, ds_X_test_sc, ds_y_test, n_repeats=5, random_state=42)
store_feature_importances(ds_X_labels, ds_gam_perm_imp.importances_mean, 'F: Generalized Additive Model (GAM)', is_directional=False, dataset='DS_Job')

# --- EXECUTE THE DS JOB CHANGE PLOT ---
plot_vims_facet_grid(dataset='DS_Job')

<Figure size 4200x5400 with 6 Axes>

In [44]:
#| label: fig-ds-pdp
#| fig-cap: Non-Linear Feature Effects for Top Continuous Drivers (Data Science GAM)

# --- MODEL F: PARTIAL DEPENDENCE PLOTS (DS JOB CHANGE) ---
# 1. Explicitly select the Top 3 CONTINUOUS features from the GAM VIMS
# Bypassing binary flags (like company_size_Unknown) because line-plot PDPs for discrete 0/1 data are visually misleading
ds_top_3_gam_features = ['city_development_index (S)', 'last_new_job (M)', 'training_hours (R)']

# Find their exact column indices in the scaled X matrix
ds_top_3_indices = [ds_X_labels.index(feat) for feat in ds_top_3_gam_features]

# 2. Create the 3x1 Grid
fig, axes = plt.subplots(nrows=3, ncols=1, figsize=(10, 15))

for i, (feat, idx) in enumerate(zip(ds_top_3_gam_features, ds_top_3_indices)):
    ax = axes[i]
    
    # Generate the grid for the specific term
    XX = ds_gam_model.generate_X_grid(term=idx)
    
    # Calculate dependence
    pdp = ds_gam_model.partial_dependence(term=idx, X=XX)
    
    # Calculate confidence intervals
    confi = ds_gam_model.confidence_intervals(XX, width=0.95)
    
    # Plotting the curve and the confidence bounds
    ax.plot(XX[:, idx], pdp, color='purple', linewidth=2)
    ax.fill_between(XX[:, idx], confi[:, 0], confi[:, 1], color='purple', alpha=0.2)
    
    # Baseline
    ax.axhline(0, color='black', linestyle='--', linewidth=1.1)
    
    # Formatting
    ax.set_title(f"PDP: {feat}", fontsize=12, fontweight='bold')
    ax.set_xlabel("Scaled Value (0 to 1)")
    ax.set_ylabel("Partial Dependence (Log-Odds)")
    
    ax.grid(True, linestyle=':', alpha=0.6)

fig.suptitle('Non-Linear Feature Effects: Top Continuous Drivers (DS GAM)', fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

<Figure size 3000x4500 with 3 Axes>

In [45]:
get_top_kill_zones(ds_tree_model, ds_X_labels, ['Stay', 'Exit'])

=== Top 3 Attrition Kill Zones Found ===

Kill Zone 1: 93.8% Attrition Concentration
Impact: 0.9/1.0 weighted score (Applies to 30 actual employees | 0.18% of workforce)
Logic Path: city_development_index (S) <= 0.62 AND experience (A) <= 1.50 AND
relevent_experience_No relevent experience (A) <= 0.50 AND training_hours (R) > 19.50
AND training_hours (R) > 47.50 AND training_hours (R) <= 86.00

Kill Zone 2: 92.9% Attrition Concentration
Impact: 0.9/1.0 weighted score (Applies to 43 actual employees | 0.26% of workforce)
Logic Path: city_development_index (S) <= 0.62 AND experience (A) > 1.50 AND
training_hours (R) <= 73.50 AND company_size_10-100 (R) > 0.50 AND training_hours (R)
> 43.00

Kill Zone 3: 92.6% Attrition Concentration
Impact: 0.9/1.0 weighted score (Applies to 36 actual employees | 0.22% of workforce)
Logic Path: city_development_index (S) <= 0.62 AND experience (A) <= 1.50 AND
relevent_experience_No relevent experience (A) <= 0.50 AND training_hours (R) <=
19.50


In [46]:
# --- PREP DATASET FOR SHAP (DS JOB CHANGE) ---
# Re-attaching labels for readability in the waterfall plots
ds_X_test_df = pd.DataFrame(ds_X_test_sc, columns=ds_X_labels)
ds_X_train_df = pd.DataFrame(ds_X_train_sc, columns=ds_X_labels)

# --- 1. MODEL A: Standard Logistic Regression ---
ds_probas_a = ds_lr_classifier.predict_proba(ds_X_test_sc)[:, 1]
ds_idx_a_min, ds_idx_a_max = np.argmin(ds_probas_a), np.argmax(ds_probas_a)
ds_explainer_a = shap.LinearExplainer(ds_lr_classifier, ds_X_train_df)
ds_shap_a = ds_explainer_a(ds_X_test_df)

# --- 2. MODEL B: Regularized Logistic Regression ---
ds_probas_b = ds_l2logreg_model.predict_proba(ds_X_test_sc)[:, 1]
ds_idx_b_min, ds_idx_b_max = np.argmin(ds_probas_b), np.argmax(ds_probas_b)
ds_explainer_b = shap.LinearExplainer(ds_l2logreg_model, ds_X_train_df)
ds_shap_b = ds_explainer_b(ds_X_test_df)

# --- 3. MODEL C: Linear SVM ---
ds_scores_c = ds_best_svm.decision_function(ds_X_test_sc)
ds_idx_c_min, ds_idx_c_max = np.argmin(ds_scores_c), np.argmax(ds_scores_c)
ds_explainer_c = shap.LinearExplainer(ds_best_svm, ds_X_train_df)
ds_shap_c = ds_explainer_c(ds_X_test_df)

# --- 4. MODEL F: Generalized Additive Model (GAM) ---
ds_probas_f = ds_gam_model.predict_proba(ds_X_test_sc)
ds_idx_f_min, ds_idx_f_max = np.argmin(ds_probas_f), np.argmax(ds_probas_f)

# Increased background to 100 for a more stable baseline on this larger dataset
ds_gam_bg = shap.sample(ds_X_train_df, 100) 
ds_explainer_f = shap.Explainer(ds_gam_model.predict, ds_gam_bg)

# Slicing just the min/max rows here, otherwise KernelExplainer takes forever to run
ds_shap_f_safe = ds_explainer_f(ds_X_test_df.iloc[[ds_idx_f_min]], silent=True)
ds_shap_f_risk = ds_explainer_f(ds_X_test_df.iloc[[ds_idx_f_max]], silent=True)

In [47]:
#| layout-ncol: 2
#| label: fig-ds-shap-safe
#| fig-cap: 'SHAP Local Interpretability: Profiles with Lowest Attrition Probability (Data Science)'
#| fig-subcap:
#|   - 'A: Std Logistic Regression - Safest'
#|   - 'B: Reg. Logistic Regression - Safest'
#|   - 'C: Linear SVM (Tuned) - Safest'
#|   - 'F: GAM - Safest'

# --- THE "SAFE" ANCHORS (Lowest Probabilities for Data Scientists) ---
# Calling the render_shap_waterfall function defined in the IBM section

render_shap_waterfall(ds_shap_a[ds_idx_a_min], "")
render_shap_waterfall(ds_shap_b[ds_idx_b_min], "")
render_shap_waterfall(ds_shap_c[ds_idx_c_min], "")
render_shap_waterfall(ds_shap_f_safe[0], "")

<Figure size 2400x2400 with 3 Axes>

<Figure size 2400x2400 with 3 Axes>

<Figure size 2400x2400 with 3 Axes>

<Figure size 2400x2400 with 3 Axes>

In [48]:
#| layout-ncol: 2
#| label: fig-ds-shap-risk
#| fig-cap: 'SHAP Local Interpretability: Profiles with Highest Attrition Probability (Data Science)'
#| fig-subcap:
#|   - 'A: Std Logistic Regression - Riskiest'
#|   - 'B: Reg. Logistic Regression - Riskiest'
#|   - 'C: Linear SVM (Tuned) - Riskiest'
#|   - 'F: GAM - Riskiest'

# --- THE "FLIGHT RISKS" (Highest Probabilities for Data Scientists) ---

render_shap_waterfall(ds_shap_a[ds_idx_a_max], "")
render_shap_waterfall(ds_shap_b[ds_idx_b_max], "")
render_shap_waterfall(ds_shap_c[ds_idx_c_max], "")
render_shap_waterfall(ds_shap_f_risk[0], "")

<Figure size 2400x2400 with 3 Axes>

<Figure size 2400x2400 with 3 Axes>

<Figure size 2400x2400 with 3 Axes>

<Figure size 2400x2400 with 3 Axes>

In [49]:
#| label: tbl-ds-perf
#| tbl-cap: Consolidated Algorithm Performance and Hyperparameters (Data Science Dataset)
# Convert the populated results dictionary into a master dataframe
ds_performance_df = pd.DataFrame(results_dict)

# Using regex to strip out numpy wrappers from the hyperparameter strings so the table looks clean
ds_performance_df['tuned hyperparameters'] = ds_performance_df['tuned hyperparameters'].apply(
    lambda x: re.sub(r"np\.\w+\(([^)]+)\)", r"\1", str(x))
)

# Rename columns to match formal Capstone rubric terminology
ds_performance_df = ds_performance_df.rename(columns={
    'specificity': 'Retention Recall (Class 0)',
    'sensitivity': 'Attrition Recall (Class 1)',
    'f1 score': 'F1-Score (Weighted)'
})

# Sort and rank the models by Attrition Recall (Our primary optimization metric)
ds_performance_df['Rank'] = ds_performance_df['Attrition Recall (Class 1)'].rank(ascending=False, method='min')
ds_performance_df = ds_performance_df.set_index('Rank').sort_index(ascending=True)

# Format to 3 decimal places for a clean, academic presentation
ds_performance_df = ds_performance_df.round(3)

print("\n=== Data Scientist Dataset: Consolidated Algorithm Performance ===")

# Convert the dataframe to a native Markdown table so Quarto automatically wraps the text
display(Markdown(ds_performance_df.to_markdown()))


=== Data Scientist Dataset: Consolidated Algorithm Performance ===


|   Rank | model name                          | tuned hyperparameters                                                                     |   Retention Recall (Class 0) |   Attrition Recall (Class 1) |   F1-Score (Weighted) |
|-------:|:------------------------------------|:------------------------------------------------------------------------------------------|-----------------------------:|-----------------------------:|----------------------:|
|      1 | D: Decision Tree                    | {'min_samples_split': 7, 'min_samples_leaf': 30, 'max_depth': 7, 'criterion': 'log_loss'} |                        0.737 |                        0.788 |                 0.765 |
|      2 | F: Generalized Additive Model (GAM) | {'lam': 0.1}                                                                              |                        0.745 |                        0.778 |                 0.768 |
|      3 | B: Regularized Logistic Regression  | {'solver': 'saga', 'penalty': 'l2', 'C': 555.556}                                         |                        0.694 |                        0.777 |                 0.733 |
|      4 | C: Linear SVM (Tuned)               | {'C': 0.0379269019073225}                                                                 |                        0.7   |                        0.77  |                 0.735 |
|      5 | E: k-Nearest Neighbors              | n_neighbors=3                                                                             |                        0.84  |                        0.464 |                 0.744 |
|      6 | A: Standard Logistic Regression     | Baseline (Untuned)                                                                        |                        0.937 |                        0.264 |                 0.735 |